In [117]:
print("Initialisation du notebook V5")

Initialisation du notebook V5


# Train Kaggle - Model V5

# PlankEye — DATA_GAN — 2×T4 — ImageNet au départ + reprise automatique

Ce notebook entraîne **PlankEye, le modèle de détection des planches et de leurs 4 coins**.

Le nouveau dataset est `data_gan` (réel + images créées par le GAN), mais le modèle entraîné ici **n'est pas un GAN**.

## Fonctionnement

### Premier lancement
S'il n'existe aucun checkpoint compatible :

- architecture PlankEye v5 ;
- backbone **MobileNetV3-Large** ;
- poids **ImageNet préentraînés** ;
- aucun ancien checkpoint PlankEye ;
- aucun warm-start ;
- départ à l'epoch 1.

### Relance
Si `last_checkpoint_v5.pt` existe et appartient à l'expérience `v5_run` :

- les poids complets PlankEye sont restaurés ;
- l'optimiseur est restauré ;
- l'EMA et le GradScaler sont restaurés ;
- les historiques sont restaurés ;
- l'entraînement reprend à `dernier_epoch + 1`.

### Sauvegarde
- `last` : à chaque epoch en local ;
- `best` : à chaque amélioration ;
- `best + last` : sauvegarde persistante Kaggle tous les 5 epochs ;
- nouvelle session : recherche automatique du checkpoint dans `/kaggle/input`, puis tentative de téléchargement depuis `max778/chekpoints-backbone18`.


## 1 — Vérifier les 2 GPU, Internet et la CLI Kaggle
## 2 — Cloner le dépôt GitHub privé

Cette cellule récupère automatiquement la dernière version de :

```text
maaxxe/Train_Kaggle_plank_Detector
└── model/
    └── model_v6.py
```

Le token reste dans **Kaggle Secrets** et n'est jamais affiché.


In [121]:
!nvidia-smi

import shutil
import socket
import subprocess
import torch

print("\n=== GPU ===")
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA absent. Active GPU T4 x2 dans Kaggle.")

n_gpus = torch.cuda.device_count()
print("Nombre de GPU :", n_gpus)

for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} | "
        f"VRAM={props.total_memory / 1024**3:.2f} Go"
    )

if n_gpus < 2:
    raise RuntimeError(
        "Ce notebook exige 2 GPU. Sélectionne GPU T4 x2 avant de continuer."
    )

print("\nOK : 2 GPU détectés.")

print("\n=== INTERNET ===")
try:
    ip = socket.gethostbyname("api.kaggle.com")
    print("api.kaggle.com ->", ip)
except Exception as exc:
    raise RuntimeError(
        "Internet/DNS Kaggle indisponible. Active Internet dans Settings."
    ) from exc

print("\n=== CLI KAGGLE ===")
kaggle_exe = shutil.which("kaggle")
if kaggle_exe is None:
    raise RuntimeError("Commande 'kaggle' introuvable.")

print("CLI :", kaggle_exe)
result = subprocess.run(
    [kaggle_exe, "datasets", "list", "-m"],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("La CLI Kaggle n'est pas authentifiée correctement.")

print("Authentification Kaggle : OK")
print("\n".join(result.stdout.splitlines()[:6]))

Mon Sep 14 14:25:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             31W /   70W |     219MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2 — Cloner le dépôt GitHub privé

Cette cellule récupère automatiquement la dernière version de :

```text
maaxxe/Train_Kaggle_plank_Detector
└── model/
    └── model_v6.py
```

Le token reste dans **Kaggle Secrets** et n'est jamais affiché.


In [122]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import subprocess
import shutil
import os

# ============================================================
# CONFIG
# ============================================================

GITHUB_USERNAME = "maaxxe"
GITHUB_REPO = "Train_Kaggle_plank_Detector"

GITHUB_REPO_DIR = Path(
    "/kaggle/working/Train_Kaggle_plank_Detector"
)

print("=" * 72)
print("CLONE GITHUB PRIVÉ")
print("=" * 72)

# ============================================================
# TOKEN GITHUB
# ============================================================

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

if not token:
    raise RuntimeError(
        "Secret GITHUB_TOKEN introuvable. "
        "Ajoute-le dans Kaggle > Add-ons/Secrets et active-le pour ce notebook."
    )

# ============================================================
# GIT ASKPASS
# ============================================================

askpass = Path("/tmp/github_askpass.sh")

askpass.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "$GITHUB_USERNAME" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)

askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_USERNAME"] = GITHUB_USERNAME
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

repo_url = (
    f"https://github.com/"
    f"{GITHUB_USERNAME}/{GITHUB_REPO}.git"
)

# ============================================================
# SUPPRESSION ANCIEN CLONE
# ============================================================

if GITHUB_REPO_DIR.exists():
    print("Suppression de l'ancien clone...")
    shutil.rmtree(GITHUB_REPO_DIR)

print("Clone       :", repo_url)
print("Destination :", GITHUB_REPO_DIR)

# ============================================================
# CLONE
# ============================================================

try:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            repo_url,
            str(GITHUB_REPO_DIR),
        ],
        env=env,
        check=True,
    )

finally:
    if askpass.exists():
        askpass.unlink()

    env.pop("GITHUB_TOKEN", None)
    del token

# ============================================================
# AFFICHAGE ARBORESCENCE
# ============================================================

def print_tree(path: Path, prefix="", max_depth=5, depth=0):

    if depth >= max_depth:
        return

    try:
        entries = sorted(
            path.iterdir(),
            key=lambda p: (p.is_file(), p.name.lower())
        )
    except PermissionError:
        return

    for i, entry in enumerate(entries):

        if entry.name == ".git":
            continue

        is_last = i == len(entries) - 1

        connector = "└── " if is_last else "├── "

        print(prefix + connector + entry.name)

        if entry.is_dir():

            extension = "    " if is_last else "│   "

            print_tree(
                entry,
                prefix + extension,
                max_depth=max_depth,
                depth=depth + 1,
            )


print()
print("=" * 72)
print("ARBORESCENCE DU REPO")
print("=" * 72)

print(GITHUB_REPO_DIR.name)
print_tree(GITHUB_REPO_DIR)

# ============================================================
# RECHERCHE AUTOMATIQUE DES model_v6.py
# ============================================================

print()
print("=" * 72)
print("FICHIERS model_v6.py TROUVÉS")
print("=" * 72)

model_files = list(
    GITHUB_REPO_DIR.rglob("model_v6.py")
)

if not model_files:
    print("Aucun model_v6.py trouvé.")
else:
    for model in model_files:
        print(model)

print()
print("Repo OK :", GITHUB_REPO_DIR.exists())
print("\nGitHub privé prêt.")

CLONE GITHUB PRIVÉ
Suppression de l'ancien clone...
Clone       : https://github.com/maaxxe/Train_Kaggle_plank_Detector.git
Destination : /kaggle/working/Train_Kaggle_plank_Detector


Cloning into '/kaggle/working/Train_Kaggle_plank_Detector'...



ARBORESCENCE DU REPO
Train_Kaggle_plank_Detector
├── GAN_PlankEYE
│   ├── config.py
│   ├── dataset.py
│   ├── GAN_PlankEye_v2_Kaggle_512.ipynb
│   ├── generate.py
│   ├── models.py
│   ├── prepare_dataset.py
│   ├── README.md
│   ├── train.py
│   └── utils.py
├── GeoNet
│   ├── model
│   │   └── multiforme_model.py
│   └── train
│       └── Train_GeoNet_Kaggle_2xT4.ipynb
├── GeoNet_paddle
│   ├── model
│   │   ├── __init__.py
│   │   └── multiforme_model.py
│   ├── model_poids
│   │   └── .gitkeep
│   ├── tools
│   │   └── export_torch_checkpoint_npz.py
│   ├── train
│   │   ├── train_geonet_baidu.py
│   │   └── Train_GeoNet_Baidu_V100.ipynb
│   └── README.md
├── Plankeye
│   ├── dataset
│   │   ├── model_improved.py
│   │   └── PlankEye_Colab.ipynb
│   ├── model
│   │   ├── model.py
│   │   ├── model_v3_.py
│   │   └── model_v5.py
│   ├── model_poids
│   │   ├── best_plankeye_v4_1class_512.pt
│   │   └── last_plankeye_v4_1class_512.pt
│   ├── train
│   │   ├── PlankEye_Training_2xT4

## 3 — Préparer le projet et le nouveau dataset `data_gan`

Cette cellule :

- copie `Plankeye/model/model_v6.py` depuis ton dépôt GitHub privé ;
- cherche automatiquement le nouveau dataset `data_gan` dans `/kaggle/input` ;
- accepte un dataset dont le dossier racine contient directement `images/` et `labels/` ;
- vérifie les associations image/label ;
- **n'importe aucun ancien checkpoint** ;
- prépare un nouvel entraînement indépendant.


In [120]:
from pathlib import Path
import os
import sys
import shutil
import re

INPUT_ROOT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")

GITHUB_REPO_DIR = Path(
    "/kaggle/working/Train_Kaggle_plank_Detector"
)
GITHUB_MODEL = (
    GITHUB_REPO_DIR
    / "Plankeye"
    / "model"
    / "model_v6.py"
)
LOCAL_MODEL = PROJECT / "model_v6.py"

# Le reste du notebook utilisera ce lien.
DATA_DIR = PROJECT / "data_kaggle_2_propre"

PROJECT.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PRÉPARATION PLANK EYE — data_kaggle_2 — FROM SCRATCH")
print("=" * 80)


# ============================================================
# 1. MODÈLE DEPUIS GITHUB
# ============================================================

print("\n=== MODÈLE ===")

if not GITHUB_REPO_DIR.exists():
    raise FileNotFoundError(
        "Repo GitHub privé introuvable :\n"
        f"{GITHUB_REPO_DIR}\n\n"
        "Exécute d'abord la cellule de clone GitHub."
    )

if not GITHUB_MODEL.exists():
    raise FileNotFoundError(
        "model/model_v6.py introuvable :\n"
        f"{GITHUB_MODEL}"
    )

shutil.copy2(GITHUB_MODEL, LOCAL_MODEL)

print("Source :", GITHUB_MODEL)
print("Copié  :", LOCAL_MODEL)
print("OK     :", LOCAL_MODEL.exists())


# ============================================================
# 2. TROUVER data_kaggle_2
# ============================================================

print("\n=== DATASET data_kaggle_2 ===")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

dataset_candidates = []

# Cherche tous les dossiers contenant images/ + labels/.
for images_dir in INPUT_ROOT.rglob("images"):
    if not images_dir.is_dir():
        continue

    candidate = images_dir.parent
    labels_dir = candidate / "labels"

    if not labels_dir.is_dir():
        continue

    path_lower = "/".join(candidate.parts).lower()

    # Priorité aux chemins dont le nom rappelle data_kaggle_2
    score = 0
    if candidate.name.lower() == "data_kaggle_2_propre":
        score += 100
    if "data_kaggle_2_propre" in path_lower:
        score += 80
    if "data_kaggle" in path_lower:
        score += 50
    if "gan" in path_lower:
        score += 20

    dataset_candidates.append((score, candidate))

if not dataset_candidates:
    raise FileNotFoundError(
        "Aucun dataset avec images/ + labels/ trouvé dans /kaggle/input."
    )

dataset_candidates.sort(
    key=lambda x: (-x[0], str(x[1]))
)

print("Candidats compatibles :")
for score, p in dataset_candidates:
    print(f"  score={score:3d} | {p}")

best_score, src = dataset_candidates[0]

if best_score <= 0 and len(dataset_candidates) > 1:
    raise RuntimeError(
        "Plusieurs datasets compatibles trouvés mais aucun ne ressemble "
        "clairement à data_kaggle_2. Vérifie les Inputs Kaggle."
    )

print("\nDataset sélectionné :")
print(" ", src)


# ============================================================
# 3. LIEN /kaggle/working/.../data_kaggle_2
# ============================================================

if DATA_DIR.exists() or DATA_DIR.is_symlink():
    if DATA_DIR.is_symlink():
        DATA_DIR.unlink()
    else:
        shutil.rmtree(DATA_DIR)

DATA_DIR.symlink_to(
    src,
    target_is_directory=True,
)

print("\nLien créé :")
print(" ", DATA_DIR)
print(" ->", src)


# ============================================================
# 4. ASSOCIATION IMAGE <-> LABEL
# Compatible :
# image12.jpg <-> image12.txt
# image12.jpg <-> label12.txt
# ============================================================

def find_label_for_image(img_path: Path):
    p = DATA_DIR / "labels" / f"{img_path.stem}.txt"
    if p.exists():
        return p

    m = re.fullmatch(r"image(\d+)", img_path.stem, flags=re.IGNORECASE)
    if m:
        p = DATA_DIR / "labels" / f"label{m.group(1)}.txt"
        if p.exists():
            return p

    return None


images = sorted(
    p for p in (DATA_DIR / "images").iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)

labels = sorted((DATA_DIR / "labels").glob("*.txt"))

pairs = []
missing = []

for img in images:
    lbl = find_label_for_image(img)
    if lbl is None:
        missing.append(img)
    else:
        pairs.append((img, lbl))

print("\n" + "=" * 80)
print("VÉRIFICATION DATASET")
print("=" * 80)

print("Images totales :", len(images))
print("Labels totaux  :", len(labels))
print("Paires valides :", len(pairs))
print("Sans label     :", len(missing))

if missing:
    print("\nPremières images sans label :")
    for p in missing[:20]:
        print(" -", p.name)

if not pairs:
    raise RuntimeError(
        "Aucune paire image/label valide trouvée dans data_kaggle_2."
    )

if len(pairs) != len(images):
    raise RuntimeError(
        f"{len(missing)} image(s) n'ont pas de label associé. "
        "Corrige le dataset avant l'entraînement."
    )

print("\n✅ data_kaggle_2 prêt")
print("Images :", DATA_DIR / "images")
print("Labels :", DATA_DIR / "labels")


# ============================================================
# 5. CHECKPOINT data_kaggle_2 : PREMIER LANCEMENT OU REPRISE
# ============================================================

import subprocess
import torch

RUN_ID = "v5_run"
KAGGLE_CKPT_DATASET = "max778/chekpoints-backbone18"

BEST = PROJECT / "best_checkpoint_v6.pt"
LAST = PROJECT / "last_checkpoint_v6.pt"
SPLIT = PROJECT / "split_v5.json"


def checkpoint_info(path: Path):

    if not path.exists():
        return None

    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        config = ckpt.get("config") or {}

        return {
            "epoch": int(ckpt.get("epoch", -1)),
            "run_id": config.get("run_id"),
            "pretrained_backbone": config.get("pretrained_backbone"),
        }

    except Exception as exc:
        print("Checkpoint illisible :", path, "|", exc)
        return None


def checkpoint_compatible(path: Path):

    info = checkpoint_info(path)

    return (
        info is not None
        and info["run_id"] == RUN_ID
        and info["pretrained_backbone"] is True
    )


def import_pair(last_src: Path):

    if not checkpoint_compatible(last_src):
        return False

    info = checkpoint_info(last_src)

    shutil.copy2(
        last_src,
        LAST,
    )

    best_src = last_src.parent / BEST.name

    if best_src.exists() and checkpoint_compatible(best_src):
        shutil.copy2(
            best_src,
            BEST,
        )

    print(
        f"✅ Checkpoint importé : epoch {info['epoch']}"
    )

    print(
        f"   reprise prévue : epoch {info['epoch'] + 1}"
    )

    return True


print()
print("=" * 80)
print("RECHERCHE CHECKPOINT data_kaggle_2_propre")
print("=" * 80)

resume_found = False


# ------------------------------------------------------------
# A. LAST déjà présent dans /kaggle/working
# ------------------------------------------------------------

if LAST.exists():

    if checkpoint_compatible(LAST):

        info = checkpoint_info(LAST)

        print(
            f"✅ LAST local compatible : epoch {info['epoch']}"
        )

        resume_found = True

    else:

        ignored = PROJECT / "checkpoints_ignores"
        ignored.mkdir(
            parents=True,
            exist_ok=True,
        )

        dst = ignored / LAST.name

        if dst.exists():
            dst.unlink()

        shutil.move(
            str(LAST),
            str(dst),
        )

        print(
            "⚠️ LAST local incompatible déplacé :",
            dst,
        )


# ------------------------------------------------------------
# B. Chercher dans les Inputs Kaggle
# ------------------------------------------------------------

if not resume_found:

    candidates = []

    for p in INPUT_ROOT.rglob(LAST.name):

        if checkpoint_compatible(p):

            info = checkpoint_info(p)

            candidates.append(
                (info["epoch"], p)
            )


    if candidates:

        candidates.sort(
            key=lambda x: x[0],
            reverse=True,
        )

        resume_found = import_pair(
            candidates[0][1]
        )


# ------------------------------------------------------------
# C. Sinon essayer de télécharger le dataset de checkpoints
# ------------------------------------------------------------

if not resume_found:

    kaggle_exe = shutil.which("kaggle")

    if kaggle_exe:

        dl_dir = Path(
            "/kaggle/working/checkpoint_data_kaggle_2_download"
        )

        if dl_dir.exists():
            shutil.rmtree(dl_dir)

        dl_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        print(
            "Tentative téléchargement :",
            KAGGLE_CKPT_DATASET,
        )

        result = subprocess.run(
            [
                kaggle_exe,
                "datasets",
                "download",
                "-d",
                KAGGLE_CKPT_DATASET,
                "-p",
                str(dl_dir),
                "--unzip",
            ],
            text=True,
            capture_output=True,
            check=False,
        )

        if result.returncode == 0:

            downloaded = []

            for p in dl_dir.rglob(LAST.name):

                if checkpoint_compatible(p):

                    info = checkpoint_info(p)

                    downloaded.append(
                        (info["epoch"], p)
                    )


            if downloaded:

                downloaded.sort(
                    key=lambda x: x[0],
                    reverse=True,
                )

                resume_found = import_pair(
                    downloaded[0][1]
                )

        else:

            print(
                "Pas de checkpoint persistant récupérable "
                "(normal au tout premier lancement)."
            )


print()
print("=" * 80)
print("MODE DE DÉMARRAGE")
print("=" * 80)

print("Run ID :", RUN_ID)
print("Best   :", BEST)
print("Last   :", LAST)
print("Split  :", SPLIT)

if resume_found:

    info = checkpoint_info(LAST)

    print()
    print("✅ REPRISE")
    print("Dernier epoch :", info["epoch"])
    print("Prochain epoch:", info["epoch"] + 1)

else:

    print()
    print("✅ PREMIER LANCEMENT")
    print("Départ        : epoch 1")
    print("Backbone      : resnet18")
    print("Poids initiaux: ImageNet préentraînés")
    print("Ancien PlankEye / warm-start : NON")

PRÉPARATION PLANK EYE — data_kaggle_2 — FROM SCRATCH

=== MODÈLE ===


FileNotFoundError: model/model_v6.py introuvable :
/kaggle/working/Train_Kaggle_plank_Detector/Plankeye/model/model_v6.py

In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")

BEST = PROJECT / "best_checkpoint_v6.pt"
LAST = PROJECT / "last_checkpoint_v6.pt"

print("=" * 70)
print("ÉTAT DES CHECKPOINTS")
print("=" * 70)

# ------------------------------------------------------------
# LAST
# ------------------------------------------------------------
if LAST.exists():
    ckpt = torch.load(
        LAST,
        map_location="cpu",
        weights_only=False,
    )

    print(f"\n✅ LAST trouvé : {LAST.name}")
    print("Dernier epoch :", ckpt.get("epoch"))
    print("Best epoch    :", ckpt.get("best_epoch"))
    print("Best metric   :", ckpt.get("best_metric"))

else:
    print("\n❌ LAST introuvable")


# ------------------------------------------------------------
# BEST
# ------------------------------------------------------------
if BEST.exists():
    ckpt_best = torch.load(
        BEST,
        map_location="cpu",
        weights_only=False,
    )

    print(f"\n✅ BEST trouvé : {BEST.name}")
    print("Epoch du BEST :", ckpt_best.get("epoch"))
    print("Best epoch    :", ckpt_best.get("best_epoch"))
    print("Best metric   :", ckpt_best.get("best_metric"))

else:
    print(f"\n⚠️ {BEST.name} introuvable")
    print("→ Le best_epoch affiché depuis LAST reste utilisable.")

In [ ]:
from pathlib import Path
import torch

ROOT = Path("/kaggle")

print("=" * 110)
print("ANALYSE DE TOUS LES FICHIERS .pt")
print("=" * 110)

pt_files = sorted(
    p for p in ROOT.rglob("*.pt")
    if p.is_file()
)

if not pt_files:
    print("❌ Aucun fichier .pt trouvé.")

for i, path in enumerate(pt_files, 1):

    size_mb = path.stat().st_size / 1024**2

    print("\n" + "=" * 110)
    print(f"[{i}/{len(pt_files)}] {path.name}")
    print(f"Chemin      : {path}")
    print(f"Taille      : {size_mb:.2f} MB")

    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        if isinstance(ckpt, dict):
            print(f"Epoch       : {ckpt.get('epoch', 'N/A')}")
            print(f"Best epoch  : {ckpt.get('best_epoch', 'N/A')}")
            print(f"Best metric : {ckpt.get('best_metric', 'N/A')}")

            config = ckpt.get("config") or {}

            if config:
                print(f"Run ID      : {config.get('run_id', 'N/A')}")
        else:
            print("⚠️ Ce .pt n'est pas un checkpoint dictionnaire.")

    except Exception as exc:
        print(f"⚠️ Impossible de lire le checkpoint : {exc}")

In [ ]:
stop

# montrer label 


In [ ]:
verif = False
if verif:
        
        # ============================================================
        # AFFICHER N IMAGES ALEATOIRES AVEC LEURS LABELS
        # ============================================================
        
        from pathlib import Path
        from PIL import Image
        import matplotlib.pyplot as plt
        import numpy as np
        import random
        
        # ============================================================
        # CONFIGURATION
        # ============================================================
        
        DATA_DIR = Path("/kaggle/working/PlankEyev2_multipieces/data_kaggle_2")
        IMAGES_DIR = DATA_DIR / "images"
        LABELS_DIR = DATA_DIR / "labels"
        
        # Nombre d'images à afficher
        N_IMAGES = 10
        
        # Graine aléatoire
        # Mettre None pour avoir des images différentes à chaque lancement
        RANDOM_SEED = None
        
        if RANDOM_SEED is not None:
            random.seed(RANDOM_SEED)
        
        
        # ============================================================
        # CHERCHER LES IMAGES AYANT UN LABEL
        # ============================================================
        
        valid_images = []
        
        for image_path in IMAGES_DIR.glob("image*.*"):
        
            if image_path.suffix.lower() not in [
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp",
                ".webp"
            ]:
                continue
        
            # Récupérer l'ID
            name = image_path.stem
        
            if not name.startswith("image"):
                continue
        
            image_id = name.replace("image", "")
        
            if not image_id.isdigit():
                continue
        
            label_path = LABELS_DIR / f"label{image_id}.txt"
        
            # On garde uniquement les images avec un label
            if label_path.exists():
                valid_images.append((image_path, label_path, int(image_id)))
        
        
        # ============================================================
        # VERIFICATION
        # ============================================================
        
        if not valid_images:
            raise FileNotFoundError(
                f"Aucune paire image + label trouvée dans :\n{IMAGES_DIR}"
            )
        
        print("=" * 70)
        print(f"Images avec label disponibles : {len(valid_images)}")
        print("=" * 70)
        
        
        # ============================================================
        # SELECTION ALEATOIRE
        # ============================================================
        
        N_IMAGES = min(N_IMAGES, len(valid_images))
        
        selected_images = random.sample(
            valid_images,
            N_IMAGES
        )
        
        
        # ============================================================
        # AFFICHER LES IMAGES
        # ============================================================
        
        for display_index, (image_path, label_path, image_id) in enumerate(
            selected_images,
            start=1
        ):
        
            print()
            print("=" * 70)
            print(
                f"IMAGE {display_index}/{N_IMAGES} "
                f"→ image{image_id}"
            )
            print(f"Image : {image_path}")
            print(f"Label : {label_path}")
            print("=" * 70)
        
            # --------------------------------------------------------
            # Charger l'image
            # --------------------------------------------------------
        
            image = Image.open(image_path).convert("RGB")
            image_np = np.array(image)
        
            height, width = image_np.shape[:2]
        
            # --------------------------------------------------------
            # Lire le label
            #
            # Format :
            # 0 x1 y1 x2 y2 x3 y3 x4 y4
            #
            # Coordonnées normalisées entre 0 et 1
            # --------------------------------------------------------
        
            objects = []
        
            with open(label_path, "r") as f:
        
                for line_number, line in enumerate(f, start=1):
        
                    line = line.strip()
        
                    if not line:
                        continue
        
                    values = line.split()
        
                    if len(values) != 9:
        
                        print(
                            f"ATTENTION ligne {line_number} : "
                            f"{len(values)} valeurs au lieu de 9"
                        )
        
                        continue
        
                    classe = int(values[0])
        
                    coords = list(map(float, values[1:]))
        
                    points = []
        
                    for i in range(0, 8, 2):
        
                        x_norm = coords[i]
                        y_norm = coords[i + 1]
        
                        x = x_norm * width
                        y = y_norm * height
        
                        points.append((x, y))
        
                    objects.append(points)
        
            # --------------------------------------------------------
            # AFFICHAGE
            # --------------------------------------------------------
        
            plt.figure(figsize=(16, 10))
        
            plt.imshow(image_np)
        
            for obj_index, points in enumerate(objects, start=1):
        
                # Fermer le polygone
                polygon = points + [points[0]]
        
                xs = [p[0] for p in polygon]
                ys = [p[1] for p in polygon]
        
                plt.plot(
                    xs,
                    ys,
                    linewidth=2
                )
        
                # ----------------------------------------------------
                # Afficher les 4 coins
                # ----------------------------------------------------
        
                for point_index, (x, y) in enumerate(
                    points,
                    start=1
                ):
        
                    plt.scatter(
                        x,
                        y,
                        s=80
                    )
        
                    plt.text(
                        x + 8,
                        y - 8,
                        f"P{point_index}",
                        fontsize=12,
                        fontweight="bold"
                    )
        
                # ----------------------------------------------------
                # Numéro de la planche au centre
                # ----------------------------------------------------
        
                center_x = sum(
                    p[0] for p in points
                ) / len(points)
        
                center_y = sum(
                    p[1] for p in points
                ) / len(points)
        
                plt.text(
                    center_x,
                    center_y,
                    f"Planche {obj_index}",
                    fontsize=14,
                    fontweight="bold"
                )
        
            # --------------------------------------------------------
            # TITRE
            # --------------------------------------------------------
        
            plt.title(
                f"Image {image_id} — "
                f"{len(objects)} planche(s) — "
                f"label{image_id}.txt"
            )
        
            plt.axis("off")
            plt.tight_layout()
            plt.show()
        
            # --------------------------------------------------------
            # INFORMATIONS
            # --------------------------------------------------------
        
            print()
            print(f"Dimensions image : {width} x {height}")
            print(f"Nombre de planches : {len(objects)}")
        
            for i, points in enumerate(objects, start=1):
        
                print(f"\nPlanche {i} :")
        
                for j, (x, y) in enumerate(
                    points,
                    start=1
                ):
        
                    print(
                        f"  P{j} : "
                        f"x={x:.1f}  "
                        f"y={y:.1f}"
                    )
else:
            print("ok")

In [ ]:
if verif:
        # ============================================================
        # VERIFICATION VISUELLE D'UN LABEL — meme logique que
        # train_plankeye_ddp.py (read_label + reorder_corners + letterbox_resize)
        # Cellule autonome, a coller telle quelle.
        # ============================================================
        
        from pathlib import Path
        from PIL import Image
        import matplotlib.pyplot as plt
        import numpy as np
        
        # ------------------------------------------------------------
        # CONFIG — adapte le chemin a ton environnement
        # ------------------------------------------------------------
        DATA_DIR = Path("/kaggle/working/PlankEyev2_multipieces/data_kaggle_2")
        # DATA_DIR = Path("/home/mrobin/MonProjet/augmentation_images_reel/data_kaggle_2")  # WSL local
        
        IMAGES_DIR = DATA_DIR / "images"
        LABELS_DIR = DATA_DIR / "labels"
        
        IMAGE_ID = "7199"
        IMG_SIZE = 512  # doit matcher IMG_SIZE de model_v6.py
        
        # ------------------------------------------------------------
        # FONCTIONS COPIEES TELLES QUELLES DE train_plankeye_ddp.py
        # ------------------------------------------------------------
        
        def reorder_corners(pts: np.ndarray) -> np.ndarray:
            pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
            center = pts.mean(axis=0)
            angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
            return pts[np.argsort(angles)].astype(np.float32)
        
        
        def polygon_area_np(pts: np.ndarray) -> float:
            pts = np.asarray(pts, dtype=np.float64)
            x, y = pts[:, 0], pts[:, 1]
            return 0.5 * abs(float(np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)))
        
        
        def read_label(label_path: Path):
            """Identique a train_plankeye_ddp.py : reorder + filtre aire quasi nulle."""
            objects = []
            with open(label_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 9:
                        continue
                    try:
                        coords = np.asarray([float(v) for v in parts[1:]], dtype=np.float32)
                    except ValueError:
                        continue
                    if not np.isfinite(coords).all():
                        continue
                    pts = coords.reshape(4, 2)
                    if (pts < -0.02).any() or (pts > 1.02).any():
                        continue
                    pts = np.clip(pts, 0.0, 1.0)
                    pts = reorder_corners(pts)
                    if polygon_area_np(pts) < 1e-06:
                        continue
                    objects.append({"cls": 0, "corners": pts})
            return objects
        
        
        def letterbox_resize(image, target_size, objects):
            """Identique a train_plankeye_ddp.py."""
            w, h = image.size
            scale = min(target_size / w, target_size / h)
            new_w = max(1, int(round(w * scale)))
            new_h = max(1, int(round(h * scale)))
            scale_x, scale_y = new_w / w, new_h / h
            left = (target_size - new_w) // 2
            top = (target_size - new_h) // 2
            resampling = getattr(Image, "Resampling", Image)
            resized = image.resize((new_w, new_h), resample=resampling.BILINEAR)
            canvas = Image.new("RGB", (target_size, target_size), (114, 114, 114))
            canvas.paste(resized, (left, top))
            transformed = []
            for obj in objects:
                pts = np.asarray(obj["corners"], dtype=np.float32).copy()
                px = pts[:, 0] * w * scale_x + left
                py = pts[:, 1] * h * scale_y + top
                out = np.stack([px / target_size, py / target_size], axis=1)
                transformed.append({"cls": int(obj["cls"]), "corners": reorder_corners(out)})
            return canvas, transformed
        
        
        # ------------------------------------------------------------
        # TROUVER IMAGE + LABEL (meme logique que collect_pairs)
        # ------------------------------------------------------------
        
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            candidate = IMAGES_DIR / f"image{IMAGE_ID}{ext}"
            if candidate.exists():
                img_path = candidate
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Aucune image trouvee pour l'id {IMAGE_ID} dans {IMAGES_DIR}")
        
        label_path = LABELS_DIR / f"{img_path.stem}.txt"
        if not label_path.exists():
            label_path = LABELS_DIR / f"label{IMAGE_ID}.txt"
        if not label_path.exists():
            raise FileNotFoundError(f"Aucun label trouve pour l'id {IMAGE_ID} dans {LABELS_DIR}")
        
        print(f"Image : {img_path}")
        print(f"Label : {label_path}")
        
        # ------------------------------------------------------------
        # LECTURE BRUTE (ordre du fichier, sans reorder) pour comparaison
        # ------------------------------------------------------------
        
        raw_objects = []
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 9:
                    continue
                coords = list(map(float, parts[1:]))
                raw_objects.append(np.array(coords, dtype=np.float32).reshape(4, 2))
        
        # ------------------------------------------------------------
        # LECTURE + TRANSFORM (logique exacte du notebook)
        # ------------------------------------------------------------
        
        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        
        objects = read_label(label_path)
        canvas, canvas_objects = letterbox_resize(image, IMG_SIZE, objects)
        
        # ------------------------------------------------------------
        # AFFICHAGE — 3 vues cote a cote
        # ------------------------------------------------------------
        
        fig, axes = plt.subplots(1, 3, figsize=(22, 8))
        
        # Vue 1 : ordre brut du .txt (comme tes scripts de visu actuels)
        axes[0].imshow(np.array(image))
        for pts in raw_objects:
            px, py = pts[:, 0] * w, pts[:, 1] * h
            px, py = np.append(px, px[0]), np.append(py, py[0])
            axes[0].plot(px, py, linewidth=2, color="red")
            for i, (x, y) in enumerate(pts, start=1):
                axes[0].scatter(x * w, y * h, s=60)
                axes[0].text(x * w + 6, y * h - 6, str(i), fontsize=10, fontweight="bold")
        axes[0].set_title("Ordre BRUT du .txt")
        axes[0].axis("off")
        
        # Vue 2 : apres reorder_corners (read_label du notebook)
        axes[1].imshow(np.array(image))
        for obj in objects:
            pts = obj["corners"]
            px, py = pts[:, 0] * w, pts[:, 1] * h
            px, py = np.append(px, px[0]), np.append(py, py[0])
            axes[1].plot(px, py, linewidth=2, color="lime")
            for i, (x, y) in enumerate(pts, start=1):
                axes[1].scatter(x * w, y * h, s=60)
                axes[1].text(x * w + 6, y * h - 6, str(i), fontsize=10, fontweight="bold")
        axes[1].set_title("Apres reorder_corners (read_label)")
        axes[1].axis("off")
        
        # Vue 3 : apres letterbox_resize (input reel du modele)
        axes[2].imshow(np.array(canvas))
        for obj in canvas_objects:
            pts = obj["corners"] * IMG_SIZE
            px = np.append(pts[:, 0], pts[0, 0])
            py = np.append(pts[:, 1], pts[0, 1])
            axes[2].plot(px, py, linewidth=2, color="cyan")
        axes[2].set_title(f"Apres letterbox_resize ({IMG_SIZE}x{IMG_SIZE}) — input reel du modele")
        axes[2].axis("off")
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nDimensions image originale : {w}x{h}")
        print(f"Nb objets lus (apres filtre read_label) : {len(objects)}")
else:
    print("ok")

In [ ]:
if verif:
        # ============================================================
        # VERIFICATION COMPLETE DES LABELS - VERSION CPU / CUDA
        # Une seule cellule Kaggle
        # ============================================================
        
        from pathlib import Path
        from tqdm.auto import tqdm
        from PIL import Image
        import matplotlib.pyplot as plt
        import numpy as np
        import torch
        import random
        import math
        from collections import Counter, defaultdict
        
        
        # ============================================================
        # CONFIGURATION
        # ============================================================
        
        DATA_DIR = Path(
            "/kaggle/working/PlankEyev2_multipieces/data_kaggle_2"
        )
        
        IMAGES_DIR = DATA_DIR / "images"
        LABELS_DIR = DATA_DIR / "labels"
        
        # Nombre maximal d'images suspectes à afficher
        MAX_IMAGES = 30
        
        # None = ordre aléatoire
        # 42 = toujours les mêmes images
        RANDOM_SEED = 42
        
        # Taille des lots envoyés au GPU
        BATCH_SIZE = 8192
        
        # Classe attendue
        VALID_CLASSES = {0}
        
        # Utiliser CUDA automatiquement si disponible
        DEVICE = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )
        
        if RANDOM_SEED is not None:
            random.seed(RANDOM_SEED)
        
        
        # ============================================================
        # SEUILS
        # ============================================================
        
        # Angles normaux d'une planche approximativement rectangulaire
        ANGLE_MIN = 70.0
        ANGLE_MAX = 110.0
        
        # Angle extrêmement anormal
        EXTREME_ANGLE_MIN = 20.0
        EXTREME_ANGLE_MAX = 160.0
        
        # Différence tolérée sur la somme des angles
        MAX_ANGLE_SUM_ERROR = 8.0
        
        # Distance minimale entre deux points
        MIN_POINT_DISTANCE_RATIO = 0.015
        
        # Longueur minimale d'un côté / diagonale image
        MIN_SIDE_IMAGE_RATIO = 0.010
        
        # Longueur minimale d'un côté / diagonale objet
        MIN_SIDE_DIAGONAL_RATIO = 0.10
        
        # Aire quadrilatère / image
        MIN_AREA_RATIO = 0.003
        MAX_AREA_RATIO = 0.95
        
        # Aire quadrilatère / bounding box
        MIN_FILL_RATIO = 0.30
        
        # Comparaison côtés opposés
        MAX_OPPOSITE_SIDE_RATIO = 1.60
        
        # Comparaison diagonales
        MAX_DIAGONAL_RATIO = 1.60
        
        # Ratio largeur / hauteur bounding box
        MAX_ASPECT_RATIO = 15.0
        
        # Tolérance parallélisme
        MAX_PARALLEL_ERROR = 20.0
        
        # Distance point -> côté opposé
        MIN_POINT_OPPOSITE_SIDE_RATIO = 0.02
        
        # Aire minimale de 3 points successifs
        MIN_TRIANGLE_AREA_RATIO = 0.001
        
        # Décalage maximum entre milieux des diagonales
        MAX_DIAGONAL_CENTER_ERROR_RATIO = 0.20
        
        # Si un quadrilatère est extrêmement fin
        MIN_MIN_ALTITUDE_RATIO = 0.025
        
        # Différence maximale entre les deux paires d'angles opposés
        MAX_OPPOSITE_ANGLE_DIFF = 30.0
        
        # Différence entre côtés parallèles
        MAX_PARALLEL_SIDE_ANGLE = 20.0
        
        
        # ============================================================
        # INFORMATIONS
        # ============================================================
        
        print("=" * 80)
        print("VERIFICATION AUTOMATIQUE DES LABELS")
        print("=" * 80)
        print(f"Dataset : {DATA_DIR}")
        print(f"Device  : {DEVICE}")
        
        if DEVICE.type == "cuda":
            print(f"GPU     : {torch.cuda.get_device_name(0)}")
        
        print()
        
        
        # ============================================================
        # TROUVER LES IMAGES
        # ============================================================
        
        valid_extensions = {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp"
        }
        
        image_map = {}
        
        for image_path in IMAGES_DIR.glob("image*.*"):
        
            if image_path.suffix.lower() not in valid_extensions:
                continue
        
            stem = image_path.stem
        
            if not stem.startswith("image"):
                continue
        
            image_id = stem[5:]
        
            if not image_id.isdigit():
                continue
        
            image_map[int(image_id)] = image_path
        
        
        print(f"Images trouvées : {len(image_map)}")
        
        
        # ============================================================
        # VERIFIER LES LABELS MANQUANTS / ORPHELINS
        # ============================================================
        
        missing_labels = []
        
        for image_id, image_path in image_map.items():
        
            label_path = LABELS_DIR / f"label{image_id}.txt"
        
            if not label_path.exists():
                missing_labels.append(
                    (image_id, image_path)
                )
        
        
        orphan_labels = []
        
        for label_path in LABELS_DIR.glob("label*.txt"):
        
            label_id = label_path.stem[5:]
        
            if (
                label_id.isdigit()
                and int(label_id) not in image_map
            ):
                orphan_labels.append(
                    label_path
                )
        
        
        # ============================================================
        # PARSING DE TOUS LES LABELS
        # ============================================================
        
        records = []
        
        parse_errors = defaultdict(list)
        
        all_image_ids = sorted(image_map.keys())
        
        
        for image_id in tqdm(
            all_image_ids,
            desc="Lecture images / labels",
            unit="image"
        ):
        
            image_path = image_map[image_id]
        
            label_path = LABELS_DIR / f"label{image_id}.txt"
        
            if not label_path.exists():
                continue
        
            # --------------------------------------------------------
            # Taille image
            # --------------------------------------------------------
        
            try:
        
                with Image.open(image_path) as im:
                    width, height = im.size
        
            except Exception as e:
        
                parse_errors[image_id].append(
                    f"image illisible : {e}"
                )
        
                continue
        
            # --------------------------------------------------------
            # Lire le label
            # --------------------------------------------------------
        
            try:
        
                lines = label_path.read_text().splitlines()
        
            except Exception as e:
        
                parse_errors[image_id].append(
                    f"label illisible : {e}"
                )
        
                continue
        
        
            if not any(line.strip() for line in lines):
        
                parse_errors[image_id].append(
                    "label vide"
                )
        
                continue
        
        
            for line_number, line in enumerate(
                lines,
                start=1
            ):
        
                line = line.strip()
        
                if not line:
                    continue
        
                values = line.split()
        
                # ====================================================
                # NOMBRE DE VALEURS
                # ====================================================
        
                if len(values) != 9:
        
                    parse_errors[image_id].append(
                        f"ligne {line_number} : "
                        f"{len(values)} valeurs au lieu de 9"
                    )
        
                    continue
        
        
                # ====================================================
                # CLASSE
                # ====================================================
        
                try:
                    class_id = int(values[0])
        
                except ValueError:
        
                    parse_errors[image_id].append(
                        f"ligne {line_number} : classe invalide"
                    )
        
                    continue
        
        
                if class_id not in VALID_CLASSES:
        
                    parse_errors[image_id].append(
                        f"ligne {line_number} : "
                        f"classe inconnue {class_id}"
                    )
        
        
                # ====================================================
                # COORDONNEES
                # ====================================================
        
                try:
        
                    coords = np.array(
                        list(
                            map(
                                float,
                                values[1:]
                            )
                        ),
                        dtype=np.float32
                    ).reshape(4, 2)
        
                except Exception:
        
                    parse_errors[image_id].append(
                        f"ligne {line_number} : coordonnées invalides"
                    )
        
                    continue
        
        
                # ====================================================
                # NaN / inf
                # ====================================================
        
                if not np.isfinite(coords).all():
        
                    parse_errors[image_id].append(
                        f"ligne {line_number} : NaN ou inf"
                    )
        
                    continue
        
        
                # ====================================================
                # Coordonnées hors [0,1]
                # ====================================================
        
                bad_coords = np.logical_or(
                    coords < 0.0,
                    coords > 1.0
                )
        
                if bad_coords.any():
        
                    bad_indices = np.argwhere(
                        bad_coords
                    )
        
                    for point_idx, coord_idx in bad_indices:
        
                        coord_name = (
                            "x"
                            if coord_idx == 0
                            else "y"
                        )
        
                        parse_errors[image_id].append(
                            f"ligne {line_number} : "
                            f"P{point_idx + 1} {coord_name} "
                            f"hors [0,1] = "
                            f"{coords[point_idx, coord_idx]:.5f}"
                        )
        
        
                # ====================================================
                # Pixel coordinates
                # ====================================================
        
                points = coords.copy()
        
                points[:, 0] *= width
                points[:, 1] *= height
        
        
                records.append(
                    {
                        "image_id": image_id,
                        "image_path": image_path,
                        "label_path": label_path,
                        "line_number": line_number,
                        "class_id": class_id,
                        "width": width,
                        "height": height,
                        "points": points,
                        "norm_points": coords,
                    }
                )
        
        
        print()
        print(f"Quadrilatères valides à tester : {len(records)}")
        print()
        
        
        # ============================================================
        # RESULTATS
        # ============================================================
        
        object_errors = defaultdict(list)
        
        
        def add_error(record_index, message):
        
            object_errors[record_index].append(
                message
            )
        
        
        # ============================================================
        # CALCULS GEOMETRIQUES CUDA / CPU
        # ============================================================
        
        EPS = 1e-8
        
        
        for batch_start in tqdm(
            range(0, len(records), BATCH_SIZE),
            desc=f"Géométrie ({DEVICE.type.upper()})",
            unit="batch"
        ):
        
            batch_records = records[
                batch_start:
                batch_start + BATCH_SIZE
            ]
        
            # --------------------------------------------------------
            # Tensor points
            # [B, 4, 2]
            # --------------------------------------------------------
        
            points_np = np.stack(
                [
                    r["points"]
                    for r in batch_records
                ],
                axis=0
            )
        
            points = torch.tensor(
                points_np,
                dtype=torch.float32,
                device=DEVICE
            )
        
            B = points.shape[0]
        
        
            widths = torch.tensor(
                [
                    r["width"]
                    for r in batch_records
                ],
                dtype=torch.float32,
                device=DEVICE
            )
        
            heights = torch.tensor(
                [
                    r["height"]
                    for r in batch_records
                ],
                dtype=torch.float32,
                device=DEVICE
            )
        
        
            image_diag = torch.sqrt(
                widths ** 2
                +
                heights ** 2
            )
        
            image_area = (
                widths
                *
                heights
            )
        
        
            # ========================================================
            # COTES
            # ========================================================
        
            next_points = torch.roll(
                points,
                shifts=-1,
                dims=1
            )
        
            side_vectors = (
                next_points
                -
                points
            )
        
            sides = torch.linalg.norm(
                side_vectors,
                dim=2
            )
        
        
            # ========================================================
            # DIAGONALES
            # ========================================================
        
            diag1 = torch.linalg.norm(
                points[:, 2] - points[:, 0],
                dim=1
            )
        
            diag2 = torch.linalg.norm(
                points[:, 3] - points[:, 1],
                dim=1
            )
        
            diagonals = torch.stack(
                [
                    diag1,
                    diag2
                ],
                dim=1
            )
        
        
            # ========================================================
            # DISTANCES ENTRE TOUS LES POINTS
            # ========================================================
        
            pair_dist = torch.cdist(
                points,
                points
            )
        
        
            # ========================================================
            # ANGLES
            # ========================================================
        
            previous_points = torch.roll(
                points,
                shifts=1,
                dims=1
            )
        
            next_points = torch.roll(
                points,
                shifts=-1,
                dims=1
            )
        
            v1 = (
                previous_points
                -
                points
            )
        
            v2 = (
                next_points
                -
                points
            )
        
            v1_norm = torch.linalg.norm(
                v1,
                dim=2
            )
        
            v2_norm = torch.linalg.norm(
                v2,
                dim=2
            )
        
            dot = (
                v1
                *
                v2
            ).sum(dim=2)
        
            cos_angle = (
                dot
                /
                (
                    v1_norm
                    *
                    v2_norm
                    +
                    EPS
                )
            )
        
            cos_angle = torch.clamp(
                cos_angle,
                -1.0,
                1.0
            )
        
            angles = torch.rad2deg(
                torch.acos(
                    cos_angle
                )
            )
        
        
            # ========================================================
            # AIRE DU POLYGONE
            # ========================================================
        
            x = points[:, :, 0]
            y = points[:, :, 1]
        
            x_next = torch.roll(
                x,
                shifts=-1,
                dims=1
            )
        
            y_next = torch.roll(
                y,
                shifts=-1,
                dims=1
            )
        
            signed_area = 0.5 * (
                (
                    x * y_next
                    -
                    x_next * y
                ).sum(dim=1)
            )
        
            area = torch.abs(
                signed_area
            )
        
            area_ratio = (
                area
                /
                (image_area + EPS)
            )
        
        
            # ========================================================
            # BOUNDING BOX
            # ========================================================
        
            min_x = x.min(dim=1).values
            max_x = x.max(dim=1).values
        
            min_y = y.min(dim=1).values
            max_y = y.max(dim=1).values
        
            bbox_w = (
                max_x
                -
                min_x
            )
        
            bbox_h = (
                max_y
                -
                min_y
            )
        
            bbox_area = (
                bbox_w
                *
                bbox_h
            )
        
            aspect = torch.maximum(
                bbox_w / (bbox_h + EPS),
                bbox_h / (bbox_w + EPS)
            )
        
            fill_ratio = (
                area
                /
                (bbox_area + EPS)
            )
        
        
            # ========================================================
            # CROSS PRODUCTS => CONVEXITE
            # ========================================================
        
            edge1 = torch.roll(
                points,
                shifts=-1,
                dims=1
            ) - points
        
            edge2 = torch.roll(
                points,
                shifts=-2,
                dims=1
            ) - torch.roll(
                points,
                shifts=-1,
                dims=1
            )
        
            cross = (
                edge1[:, :, 0]
                *
                edge2[:, :, 1]
                -
                edge1[:, :, 1]
                *
                edge2[:, :, 0]
            )
        
            convex_positive = (
                cross > 0
            ).all(dim=1)
        
            convex_negative = (
                cross < 0
            ).all(dim=1)
        
            convex = (
                convex_positive
                |
                convex_negative
            )
        
        
            # ========================================================
            # AIRES TRIANGLES SUCCESSIFS
            # ========================================================
        
            A = points
        
            Bp = torch.roll(
                points,
                shifts=-1,
                dims=1
            )
        
            C = torch.roll(
                points,
                shifts=-2,
                dims=1
            )
        
            triangle_area = torch.abs(
                (
                    A[:, :, 0]
                    *
                    (
                        Bp[:, :, 1]
                        -
                        C[:, :, 1]
                    )
                    +
                    Bp[:, :, 0]
                    *
                    (
                        C[:, :, 1]
                        -
                        A[:, :, 1]
                    )
                    +
                    C[:, :, 0]
                    *
                    (
                        A[:, :, 1]
                        -
                        Bp[:, :, 1]
                    )
                )
                / 2.0
            )
        
        
            # ========================================================
            # COTES OPPOSES
            # ========================================================
        
            opposite_ratio_13 = (
                torch.maximum(
                    sides[:, 0],
                    sides[:, 2]
                )
                /
                (
                    torch.minimum(
                        sides[:, 0],
                        sides[:, 2]
                    )
                    +
                    EPS
                )
            )
        
            opposite_ratio_24 = (
                torch.maximum(
                    sides[:, 1],
                    sides[:, 3]
                )
                /
                (
                    torch.minimum(
                        sides[:, 1],
                        sides[:, 3]
                    )
                    +
                    EPS
                )
            )
        
        
            # ========================================================
            # RATIO DIAGONALES
            # ========================================================
        
            diagonal_ratio = (
                torch.maximum(
                    diag1,
                    diag2
                )
                /
                (
                    torch.minimum(
                        diag1,
                        diag2
                    )
                    +
                    EPS
                )
            )
        
        
            # ========================================================
            # PARALLELISME
            # ========================================================
        
            def batch_parallel_error(v_a, v_b):
        
                norm_a = torch.linalg.norm(
                    v_a,
                    dim=1
                )
        
                norm_b = torch.linalg.norm(
                    v_b,
                    dim=1
                )
        
                cos = (
                    (
                        v_a
                        *
                        v_b
                    ).sum(dim=1)
                    /
                    (
                        norm_a
                        *
                        norm_b
                        +
                        EPS
                    )
                )
        
                cos = torch.clamp(
                    cos,
                    -1.0,
                    1.0
                )
        
                angle = torch.rad2deg(
                    torch.acos(cos)
                )
        
                return torch.minimum(
                    angle,
                    torch.abs(
                        180.0 - angle
                    )
                )
        
        
            parallel_13 = batch_parallel_error(
                side_vectors[:, 0],
                side_vectors[:, 2]
            )
        
            parallel_24 = batch_parallel_error(
                side_vectors[:, 1],
                side_vectors[:, 3]
            )
        
        
            # ========================================================
            # MILIEUX DES DIAGONALES
            # ========================================================
        
            midpoint_diag1 = (
                points[:, 0]
                +
                points[:, 2]
            ) / 2.0
        
            midpoint_diag2 = (
                points[:, 1]
                +
                points[:, 3]
            ) / 2.0
        
            midpoint_distance = torch.linalg.norm(
                midpoint_diag1
                -
                midpoint_diag2,
                dim=1
            )
        
            max_diag = torch.maximum(
                diag1,
                diag2
            )
        
            midpoint_error_ratio = (
                midpoint_distance
                /
                (
                    max_diag
                    +
                    EPS
                )
            )
        
        
            # ========================================================
            # ANGLES OPPOSES
            # ========================================================
        
            opposite_angle_error_13 = torch.abs(
                angles[:, 0]
                -
                angles[:, 2]
            )
        
            opposite_angle_error_24 = torch.abs(
                angles[:, 1]
                -
                angles[:, 3]
            )
        
        
            # ========================================================
            # TRANSFERER LES RESULTATS SUR CPU
            # ========================================================
        
            cpu = {
                "sides": sides.cpu().numpy(),
                "diagonals": diagonals.cpu().numpy(),
                "pair_dist": pair_dist.cpu().numpy(),
                "angles": angles.cpu().numpy(),
                "area_ratio": area_ratio.cpu().numpy(),
                "area": area.cpu().numpy(),
                "aspect": aspect.cpu().numpy(),
                "fill_ratio": fill_ratio.cpu().numpy(),
                "convex": convex.cpu().numpy(),
                "triangle_area": triangle_area.cpu().numpy(),
                "opp13": opposite_ratio_13.cpu().numpy(),
                "opp24": opposite_ratio_24.cpu().numpy(),
                "diag_ratio": diagonal_ratio.cpu().numpy(),
                "parallel13": parallel_13.cpu().numpy(),
                "parallel24": parallel_24.cpu().numpy(),
                "middle_error": midpoint_error_ratio.cpu().numpy(),
                "opp_angle13": opposite_angle_error_13.cpu().numpy(),
                "opp_angle24": opposite_angle_error_24.cpu().numpy(),
                "image_diag": image_diag.cpu().numpy(),
                "image_area": image_area.cpu().numpy(),
                "bbox_w": bbox_w.cpu().numpy(),
                "bbox_h": bbox_h.cpu().numpy(),
            }
        
        
            # ========================================================
            # INTERPRETATION DES RESULTATS
            # ========================================================
        
            for local_idx, record in enumerate(
                batch_records
            ):
        
                global_idx = (
                    batch_start
                    +
                    local_idx
                )
        
                points_local = record["points"]
        
                img_diag = cpu["image_diag"][local_idx]
                img_area = cpu["image_area"][local_idx]
        
                sides_local = cpu["sides"][local_idx]
                diagonals_local = cpu["diagonals"][local_idx]
                angles_local = cpu["angles"][local_idx]
        
                # ----------------------------------------------------
                # Points trop proches
                # ----------------------------------------------------
        
                min_point_dist = (
                    img_diag
                    *
                    MIN_POINT_DISTANCE_RATIO
                )
        
                for i in range(4):
        
                    for j in range(
                        i + 1,
                        4
                    ):
        
                        d = cpu["pair_dist"][
                            local_idx,
                            i,
                            j
                        ]
        
                        if d < min_point_dist:
        
                            add_error(
                                global_idx,
                                f"P{i + 1}/P{j + 1} trop proches "
                                f"({d:.1f}px)"
                            )
        
        
                # ----------------------------------------------------
                # Côtés trop petits
                # ----------------------------------------------------
        
                min_side_image = (
                    img_diag
                    *
                    MIN_SIDE_IMAGE_RATIO
                )
        
                for i, side in enumerate(
                    sides_local
                ):
        
                    if side < min_side_image:
        
                        add_error(
                            global_idx,
                            f"côté {i + 1} extrêmement court "
                            f"({side:.1f}px)"
                        )
        
        
                max_object_diag = max(
                    diagonals_local
                )
        
                if max_object_diag > 0:
        
                    for i, side in enumerate(
                        sides_local
                    ):
        
                        if (
                            side / max_object_diag
                            <
                            MIN_SIDE_DIAGONAL_RATIO
                        ):
        
                            add_error(
                                global_idx,
                                f"côté {i + 1} trop petit "
                                "par rapport aux diagonales"
                            )
        
        
                # ----------------------------------------------------
                # Angles
                # ----------------------------------------------------
        
                for i, angle in enumerate(
                    angles_local
                ):
        
                    if (
                        angle
                        <
                        EXTREME_ANGLE_MIN
                        or
                        angle
                        >
                        EXTREME_ANGLE_MAX
                    ):
        
                        add_error(
                            global_idx,
                            f"angle P{i + 1} EXTREME "
                            f"({angle:.1f}°)"
                        )
        
                    elif (
                        angle < ANGLE_MIN
                        or
                        angle > ANGLE_MAX
                    ):
        
                        add_error(
                            global_idx,
                            f"angle P{i + 1} suspect "
                            f"({angle:.1f}°)"
                        )
        
        
                angle_sum = float(
                    np.sum(
                        angles_local
                    )
                )
        
                if (
                    abs(
                        angle_sum
                        -
                        360.0
                    )
                    >
                    MAX_ANGLE_SUM_ERROR
                ):
        
                    add_error(
                        global_idx,
                        f"somme angles anormale "
                        f"({angle_sum:.1f}°)"
                    )
        
        
                # ----------------------------------------------------
                # Convexité
                # ----------------------------------------------------
        
                if not cpu["convex"][local_idx]:
        
                    add_error(
                        global_idx,
                        "quadrilatère non convexe / "
                        "ordre des coins incorrect"
                    )
        
        
                # ----------------------------------------------------
                # 3 points presque alignés
                # ----------------------------------------------------
        
                min_triangle_area = (
                    img_area
                    *
                    MIN_TRIANGLE_AREA_RATIO
                )
        
                for i, tri_area in enumerate(
                    cpu["triangle_area"][
                        local_idx
                    ]
                ):
        
                    if tri_area < min_triangle_area:
        
                        add_error(
                            global_idx,
                            f"3 points presque alignés "
                            f"autour de P{i + 1}"
                        )
        
        
                # ----------------------------------------------------
                # Côtés opposés
                # ----------------------------------------------------
        
                if (
                    cpu["opp13"][local_idx]
                    >
                    MAX_OPPOSITE_SIDE_RATIO
                ):
        
                    add_error(
                        global_idx,
                        f"côtés opposés 1/3 "
                        f"très différents "
                        f"(ratio="
                        f"{cpu['opp13'][local_idx]:.2f})"
                    )
        
        
                if (
                    cpu["opp24"][local_idx]
                    >
                    MAX_OPPOSITE_SIDE_RATIO
                ):
        
                    add_error(
                        global_idx,
                        f"côtés opposés 2/4 "
                        f"très différents "
                        f"(ratio="
                        f"{cpu['opp24'][local_idx]:.2f})"
                    )
        
        
                # ----------------------------------------------------
                # Diagonales
                # ----------------------------------------------------
        
                if (
                    cpu["diag_ratio"][local_idx]
                    >
                    MAX_DIAGONAL_RATIO
                ):
        
                    add_error(
                        global_idx,
                        f"diagonales très différentes "
                        f"(ratio="
                        f"{cpu['diag_ratio'][local_idx]:.2f})"
                    )
        
        
                # ----------------------------------------------------
                # Parallélisme
                # ----------------------------------------------------
        
                if (
                    cpu["parallel13"][local_idx]
                    >
                    MAX_PARALLEL_ERROR
                ):
        
                    add_error(
                        global_idx,
                        f"côtés 1/3 non parallèles "
                        f"(écart="
                        f"{cpu['parallel13'][local_idx]:.1f}°)"
                    )
        
        
                if (
                    cpu["parallel24"][local_idx]
                    >
                    MAX_PARALLEL_ERROR
                ):
        
                    add_error(
                        global_idx,
                        f"côtés 2/4 non parallèles "
                        f"(écart="
                        f"{cpu['parallel24'][local_idx]:.1f}°)"
                    )
        
        
                # ----------------------------------------------------
                # Angles opposés
                # ----------------------------------------------------
        
                if (
                    cpu["opp_angle13"][local_idx]
                    >
                    MAX_OPPOSITE_ANGLE_DIFF
                ):
        
                    add_error(
                        global_idx,
                        "angles opposés P1/P3 "
                        "trop différents"
                    )
        
        
                if (
                    cpu["opp_angle24"][local_idx]
                    >
                    MAX_OPPOSITE_ANGLE_DIFF
                ):
        
                    add_error(
                        global_idx,
                        "angles opposés P2/P4 "
                        "trop différents"
                    )
        
        
                # ----------------------------------------------------
                # Aire
                # ----------------------------------------------------
        
                area_ratio_local = (
                    cpu["area_ratio"][
                        local_idx
                    ]
                )
        
                if (
                    area_ratio_local
                    <
                    MIN_AREA_RATIO
                ):
        
                    add_error(
                        global_idx,
                        f"aire trop petite "
                        f"({area_ratio_local * 100:.3f}% image)"
                    )
        
        
                if (
                    area_ratio_local
                    >
                    MAX_AREA_RATIO
                ):
        
                    add_error(
                        global_idx,
                        f"aire énorme "
                        f"({area_ratio_local * 100:.1f}% image)"
                    )
        
        
                # ----------------------------------------------------
                # Aspect
                # ----------------------------------------------------
        
                if (
                    cpu["bbox_w"][local_idx] <= 1
                    or
                    cpu["bbox_h"][local_idx] <= 1
                ):
        
                    add_error(
                        global_idx,
                        "bounding box presque nulle"
                    )
        
                elif (
                    cpu["aspect"][local_idx]
                    >
                    MAX_ASPECT_RATIO
                ):
        
                    add_error(
                        global_idx,
                        f"forme extrêmement allongée "
                        f"(ratio="
                        f"{cpu['aspect'][local_idx]:.2f})"
                    )
        
        
                # ----------------------------------------------------
                # Fill ratio
                # ----------------------------------------------------
        
                if (
                    cpu["fill_ratio"][local_idx]
                    <
                    MIN_FILL_RATIO
                ):
        
                    add_error(
                        global_idx,
                        f"forme écrasée / incohérente "
                        f"(fill="
                        f"{cpu['fill_ratio'][local_idx]:.2f})"
                    )
        
        
                # ----------------------------------------------------
                # Milieux diagonales
                # ----------------------------------------------------
        
                if (
                    cpu["middle_error"][local_idx]
                    >
                    MAX_DIAGONAL_CENTER_ERROR_RATIO
                ):
        
                    add_error(
                        global_idx,
                        "milieux des diagonales "
                        "trop éloignés "
                        f"(ratio="
                        f"{cpu['middle_error'][local_idx]:.2f})"
                    )
        
        
                # ====================================================
                # CROISEMENTS DES COTES
                # CPU car seulement 2 tests / quadrilatère
                # ====================================================
        
                def orientation(a, b, c):
        
                    value = (
                        (b[1] - a[1])
                        *
                        (c[0] - b[0])
                        -
                        (b[0] - a[0])
                        *
                        (c[1] - b[1])
                    )
        
                    if abs(value) < 1e-9:
                        return 0
        
                    return (
                        1
                        if value > 0
                        else 2
                    )
        
        
                def on_segment(a, b, c):
        
                    return (
                        min(a[0], c[0])
                        <= b[0]
                        <= max(a[0], c[0])
                        and
                        min(a[1], c[1])
                        <= b[1]
                        <= max(a[1], c[1])
                    )
        
        
                def intersect(a, b, c, d):
        
                    o1 = orientation(
                        a,
                        b,
                        c
                    )
        
                    o2 = orientation(
                        a,
                        b,
                        d
                    )
        
                    o3 = orientation(
                        c,
                        d,
                        a
                    )
        
                    o4 = orientation(
                        c,
                        d,
                        b
                    )
        
                    if (
                        o1 != o2
                        and
                        o3 != o4
                    ):
                        return True
        
                    if (
                        o1 == 0
                        and
                        on_segment(
                            a,
                            c,
                            b
                        )
                    ):
                        return True
        
                    if (
                        o2 == 0
                        and
                        on_segment(
                            a,
                            d,
                            b
                        )
                    ):
                        return True
        
                    if (
                        o3 == 0
                        and
                        on_segment(
                            c,
                            a,
                            d
                        )
                    ):
                        return True
        
                    if (
                        o4 == 0
                        and
                        on_segment(
                            c,
                            b,
                            d
                        )
                    ):
                        return True
        
                    return False
        
        
                p0, p1, p2, p3 = (
                    points_local
                )
        
                if intersect(
                    p0,
                    p1,
                    p2,
                    p3
                ):
        
                    add_error(
                        global_idx,
                        "côtés 1/3 se croisent"
                    )
        
        
                if intersect(
                    p1,
                    p2,
                    p3,
                    p0
                ):
        
                    add_error(
                        global_idx,
                        "côtés 2/4 se croisent"
                    )
        
        
        # ============================================================
        # REGROUPER PAR IMAGE
        # ============================================================
        
        image_suspects = defaultdict(list)
        
        error_counter = Counter()
        
        
        for record_index, errors in object_errors.items():
        
            record = records[
                record_index
            ]
        
            image_id = record[
                "image_id"
            ]
        
            image_suspects[
                image_id
            ].append(
                (
                    record_index,
                    record,
                    errors
                )
            )
        
            for error in errors:
        
                error_type = error.split("(")[0].strip()
        
                error_counter[
                    error_type
                ] += 1
        
        
        # ============================================================
        # AJOUTER LES ERREURS DE PARSING
        # ============================================================
        
        for image_id, errors in parse_errors.items():
        
            for error in errors:
        
                error_counter[
                    error.split("(")[0].strip()
                ] += 1
        
        
        # ============================================================
        # RESULTATS
        # ============================================================
        
        print()
        print("=" * 80)
        print("RESULTATS")
        print("=" * 80)
        
        print(
            f"Images trouvées              : "
            f"{len(image_map)}"
        )
        
        print(
            f"Labels manquants             : "
            f"{len(missing_labels)}"
        )
        
        print(
            f"Labels sans image            : "
            f"{len(orphan_labels)}"
        )
        
        print(
            f"Quadrilatères analysés       : "
            f"{len(records)}"
        )
        
        print(
            f"Quadrilatères suspects       : "
            f"{len(object_errors)}"
        )
        
        all_suspect_image_ids = set(
            image_suspects.keys()
        ) | set(
            parse_errors.keys()
        )
        
        print(
            f"Images suspectes             : "
            f"{len(all_suspect_image_ids)}"
        )
        
        print()
        
        
        # ============================================================
        # TOP DES ERREURS
        # ============================================================
        
        print("=" * 80)
        print("ERREURS LES PLUS FREQUENTES")
        print("=" * 80)
        
        for error_name, count in error_counter.most_common(
            30
        ):
        
            print(
                f"{count:6d}  |  {error_name}"
            )
        
        
        # ============================================================
        # NETTOYAGE GPU
        # ============================================================
        
        if DEVICE.type == "cuda":
        
            torch.cuda.empty_cache()
        
        
        # ============================================================
        # PREPARER LES IMAGES A AFFICHER
        # ============================================================
        
        suspect_ids = list(
            all_suspect_image_ids
        )
        
        if RANDOM_SEED is not None:
            random.Random(
                RANDOM_SEED
            ).shuffle(
                suspect_ids
            )
        
        else:
            random.shuffle(
                suspect_ids
            )
        
        
        suspect_ids = suspect_ids[
            :MAX_IMAGES
        ]
        
        
        # ============================================================
        # AFFICHAGE
        # ============================================================
        
        print()
        print("=" * 80)
        print(
            f"AFFICHAGE DE "
            f"{len(suspect_ids)} "
            f"IMAGES SUSPECTES"
        )
        print("=" * 80)
        
        
        for image_id in suspect_ids:
        
            image_path = image_map.get(
                image_id
            )
        
            if image_path is None:
                continue
        
            try:
        
                image = Image.open(
                    image_path
                ).convert("RGB")
        
                image_np = np.array(
                    image
                )
        
            except Exception as e:
        
                print(
                    f"Impossible d'afficher image{image_id}: {e}"
                )
        
                continue
        
        
            height, width = image_np.shape[:2]
        
        
            plt.figure(
                figsize=(16, 10)
            )
        
            plt.imshow(
                image_np
            )
        
        
            # ========================================================
            # AFFICHER TOUS LES OBJETS
            # ========================================================
        
            records_for_image = [
                (
                    idx,
                    record
                )
                for idx, record
                in enumerate(records)
                if record["image_id"] == image_id
            ]
        
        
            for record_index, record in records_for_image:
        
                points = record[
                    "points"
                ]
        
                errors = object_errors.get(
                    record_index,
                    []
                )
        
        
                polygon = np.vstack(
                    [
                        points,
                        points[0]
                    ]
                )
        
        
                # Ligne plus épaisse si suspect
                linewidth = (
                    4
                    if errors
                    else 2
                )
        
        
                plt.plot(
                    polygon[:, 0],
                    polygon[:, 1],
                    linewidth=linewidth
                )
        
        
                # ----------------------------------------------------
                # Coins
                # ----------------------------------------------------
        
                for point_index, (
                    px,
                    py
                ) in enumerate(
                    points,
                    start=1
                ):
        
                    plt.scatter(
                        px,
                        py,
                        s=120
                    )
        
                    plt.text(
                        px + 8,
                        py - 8,
                        f"P{point_index}",
                        fontsize=11,
                        fontweight="bold"
                    )
        
        
                # ----------------------------------------------------
                # Centre
                # ----------------------------------------------------
        
                center = points.mean(
                    axis=0
                )
        
        
                if errors:
        
                    plt.text(
                        center[0],
                        center[1],
                        f"Ligne {record['line_number']}\n"
                        f"SUSPECTE",
                        fontsize=13,
                        fontweight="bold",
                        horizontalalignment="center",
                        verticalalignment="center"
                    )
        
                else:
        
                    plt.text(
                        center[0],
                        center[1],
                        f"L{record['line_number']}",
                        fontsize=10,
                        horizontalalignment="center",
                        verticalalignment="center"
                    )
        
        
            # ========================================================
            # TITRE
            # ========================================================
        
            geometry_errors = image_suspects.get(
                image_id,
                []
            )
        
            parsing = parse_errors.get(
                image_id,
                []
            )
        
        
            total_error_count = (
                sum(
                    len(errors)
                    for _, _, errors
                    in geometry_errors
                )
                +
                len(parsing)
            )
        
        
            plt.title(
                f"IMAGE {image_id} — "
                f"{total_error_count} anomalie(s)",
                fontsize=16,
                fontweight="bold"
            )
        
            plt.axis(
                "off"
            )
        
            plt.tight_layout()
        
            plt.show()
        
        
            # ========================================================
            # DETAILS TERMINAL
            # ========================================================
        
            print()
            print("=" * 80)
            print(
                f"IMAGE {image_id}"
            )
            print("=" * 80)
        
        
            if parsing:
        
                print(
                    "\nERREURS FICHIER / FORMAT :"
                )
        
                for error in parsing:
        
                    print(
                        f"  ⚠ {error}"
                    )
        
        
            for (
                record_index,
                record,
                errors
            ) in geometry_errors:
        
                print()
                print(
                    f"Ligne {record['line_number']} :"
                )
        
                for error in errors:
        
                    print(
                        f"  ⚠ {error}"
                    )
        
                print(
                    "  Coordonnées normalisées :"
                )
        
                for i, (
                    x,
                    y
                ) in enumerate(
                    record["norm_points"],
                    start=1
                ):
        
                    print(
                        f"    P{i}: "
                        f"x={x:.6f} "
                        f"y={y:.6f}"
                    )
        
        
        # ============================================================
        # LABELS MANQUANTS
        # ============================================================
        
        if missing_labels:
        
            print()
            print("=" * 80)
            print("IMAGES SANS LABEL")
            print("=" * 80)
        
            for image_id, path in missing_labels[
                :50
            ]:
        
                print(
                    f"image{image_id} -> label absent"
                )
        
            if len(missing_labels) > 50:
        
                print(
                    f"... + "
                    f"{len(missing_labels) - 50} "
                    f"autres"
                )
        
        
        # ============================================================
        # LABELS ORPHELINS
        # ============================================================
        
        if orphan_labels:
        
            print()
            print("=" * 80)
            print("LABELS SANS IMAGE")
            print("=" * 80)
        
            for path in orphan_labels[
                :50
            ]:
        
                print(
                    path.name
                )
        
            if len(orphan_labels) > 50:
        
                print(
                    f"... + "
                    f"{len(orphan_labels) - 50} "
                    f"autres"
                )
        
        
        # ============================================================
        # FIN
        # ============================================================
        
        print()
        print("=" * 80)
        print("VERIFICATION TERMINEE")
        print("=" * 80)
        
        print(
            f"Device utilisé          : {DEVICE}"
        )
        
        print(
            f"Images analysées        : {len(image_map)}"
        )
        
        print(
            f"Objets analysés         : {len(records)}"
        )
        
        print(
            f"Objets suspects         : {len(object_errors)}"
        )
        
        print(
            f"Images suspectes        : {len(all_suspect_image_ids)}"
        )
        
        print(
            f"Images affichées        : {len(suspect_ids)}"
        )
        
        print("=" * 80)
else:
        print("ok")

## 4 — Vérifier la compatibilité de `model_v6.py`


In [ ]:


import sys
import torch

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from model_v6 import NUM_CLASSES, IMG_SIZE, PlankEyeV5

print("NUM_CLASSES :", NUM_CLASSES)
print("IMG_SIZE :", IMG_SIZE)

if NUM_CLASSES != 1:
    raise RuntimeError(
        f"Le notebook attend un modèle 1 classe, NUM_CLASSES={NUM_CLASSES}"
    )

# Test de structure du modèle V5
probe = PlankEyeV5()

required_methods = (
    "forward",
)

missing = [name for name in required_methods if not hasattr(probe, name)]

if missing:
    raise RuntimeError(
        "model_v6.py incompatible. Méthodes absentes : " + ", ".join(missing)
    )

print("model_v6.py (ResNet18) compatible avec succès.")

del probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5 — Écrire le script d'entraînement DDP corrigé

%%writefile /kaggle/working/PlankEyev2_multipieces/train_plankeye_ddp.py
Le script ci-dessous contient la sauvegarde locale + l'upload automatique de `best.pt` et `last.pt` vers `max778/chekpoints-backbone18` tous les 5 epochs.

In [ ]:
%%writefile /kaggle/working/PlankEyev2_multipieces/train_plankeye_ddp.py
from __future__ import annotations

"""PlankEye V6 training script (single GPU or DDP).

Key fixes compared with the older V5 training:
- V6 absolute-corner targets; old checkpoints are intentionally incompatible semantically.
- Batch 4/GPU + gradient accumulation for a safer effective batch.
- Warmed EMA instead of an excessively slow fixed EMA.
- Dataset signature invalidates stale split manifests automatically.
- Strict label/image validation before training.
- Consistent cell-center offsets.
- Validation computes real polygon AP50/AP75, precision, recall and mean IoU.
- Best checkpoint is selected by detection quality, not only validation loss.
- Optimizer state is preserved when backbone layers are unfrozen.
"""

import copy
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
from collections import defaultdict
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
import torch
import torch.distributed as dist
import torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from tqdm.auto import tqdm

from model_v6 import (
    IMG_SIZE,
    MODEL_VERSION,
    NUM_CLASSES,
    PlankEyeV6,
    combined_loss_v6,
    decode_predictions,
    quad_iou_np,
)

# =============================================================================
# CONFIG
# =============================================================================
PROJECT = Path(os.environ.get("PLANKEYE_PROJECT", "/kaggle/working/PlankEyev2_multipieces"))
if not PROJECT.exists():
    PROJECT = Path(__file__).resolve().parent
SCRIPT_DIR = PROJECT

# Prefer the fused dataset when it exists; otherwise keep compatibility with
# the previous project layout. You can override with PLANKEYE_DATA_DIR.
_env_data = os.environ.get("PLANKEYE_DATA_DIR")
if _env_data:
    DATA_DIR = Path(_env_data)
    if not DATA_DIR.is_absolute():
        DATA_DIR = PROJECT / DATA_DIR
else:
    DATA_DIR = PROJECT / "dataset_fusionne"
    if not DATA_DIR.exists():
        DATA_DIR = PROJECT / "data_kaggle_2_propre"

IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

MODEL_DIR = SCRIPT_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = MODEL_DIR / "best_checkpoint_v6.pt"
LAST_CHECKPOINT_PATH = MODEL_DIR / "last_checkpoint_v6.pt"
SPLIT_MANIFEST_PATH = SCRIPT_DIR / "split_v6.json"
PLOT_PATH = SCRIPT_DIR / "training_curves_v6.png"

SEED = 48
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

# Safe on T4 x2. Effective batch = 4 * 2 GPUs * 2 accumulation = 16.
BATCH_SIZE = 4
VAL_BATCH_MULT = 2
GRAD_ACCUM_STEPS_DDP = 2
GRAD_ACCUM_STEPS_SINGLE = 4
EPOCHS = 150

LR_HEAD = 2.5e-4
LR_BACKBONE = 2.0e-5
MIN_LR_FACTOR = 0.08
WEIGHT_DECAY = 2.0e-4
GRAD_CLIP_NORM = 2.0

UNFREEZE_LAST_EPOCH = 5
UNFREEZE_ALL_EPOCH = 15

# Warmed EMA: early updates follow the model quickly, then smooth gradually.
EMA_MAX_DECAY = 0.999
EMA_TAU = 200.0

EARLY_STOP_PATIENCE = 30
EARLY_STOP_MIN_DELTA = 1e-4
ROT_MAX_DEG = 25.0

# Validation decoding. AP needs a low score floor; threshold metrics use 0.30.
VAL_DECODE_CONF = 0.01
VAL_REPORT_CONF = 0.30
VAL_TOPK = 50
VAL_NMS_IOU = 0.50

# Set True only if you intentionally want to delete V6 checkpoints/split.
START_FROM_SCRATCH = os.environ.get("PLANKEYE_START_FROM_SCRATCH", "0").strip() in {"1", "true", "True", "yes", "YES"}

RUN_ID = "v6_absolute_corners"
PRETRAINED_BACKBONE = True

# Optional Kaggle persistence. Empty ID disables it.
PERSIST_EVERY = 5
KAGGLE_DATASET_ID = os.environ.get("PLANKEYE_KAGGLE_DATASET", "max778/chekpoints-backbone18")
PERSIST_DIR = Path("/kaggle/working/chekpoints-backbone18_persistent")

RANK = int(os.environ.get("RANK", "0"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
WORLD_SIZE = max(1, int(os.environ.get("WORLD_SIZE", "1")))
IS_DISTRIBUTED = WORLD_SIZE > 1
IS_MAIN = RANK == 0

if torch.cuda.is_available():
    DEVICE = torch.device(f"cuda:{LOCAL_RANK}")
    DEVICE_TYPE = "cuda"
else:
    DEVICE = torch.device("cpu")
    DEVICE_TYPE = "cpu"
AMP_ENABLED = DEVICE_TYPE == "cuda"
GRAD_ACCUM_STEPS = GRAD_ACCUM_STEPS_DDP if IS_DISTRIBUTED else GRAD_ACCUM_STEPS_SINGLE


# =============================================================================
# DISTRIBUTED / SEED
# =============================================================================
def rank0_print(*args, **kwargs):
    if IS_MAIN:
        print(*args, **kwargs)


def setup_distributed() -> None:
    if DEVICE_TYPE != "cuda":
        if IS_DISTRIBUTED:
            raise RuntimeError("DDP demandé mais aucun GPU CUDA n'est disponible.")
        return
    torch.cuda.set_device(LOCAL_RANK)
    if IS_DISTRIBUTED and not dist.is_initialized():
        store_path = PROJECT / "ddp_filestore"
        # Prefer torchrun env://. The fallback FileStore matches the existing
        # Kaggle launcher; that launcher must delete a stale ddp_filestore
        # BEFORE spawning the ranks (never delete it from inside one rank).
        init_method = "env://" if os.environ.get("MASTER_ADDR") and os.environ.get("MASTER_PORT") else f"file://{store_path}"
        dist.init_process_group(
            backend="nccl",
            init_method=init_method,
            rank=RANK,
            world_size=WORLD_SIZE,
            device_id=DEVICE,
        )
    torch.backends.cudnn.benchmark = True


def cleanup_distributed() -> None:
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


def barrier() -> None:
    if IS_DISTRIBUTED and dist.is_initialized():
        dist.barrier()


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, DDP) else model


def wrap_for_training(model: nn.Module) -> nn.Module:
    if not IS_DISTRIBUTED:
        return model
    return DDP(
        model,
        device_ids=[LOCAL_RANK],
        output_device=LOCAL_RANK,
        broadcast_buffers=True,
        find_unused_parameters=False,
        gradient_as_bucket_view=True,
    )


def broadcast_stop(stop: bool) -> bool:
    if not IS_DISTRIBUTED:
        return bool(stop)
    flag = torch.tensor([1 if stop else 0], device=DEVICE, dtype=torch.int32)
    dist.broadcast(flag, src=0)
    return bool(flag.item())


def set_seed(seed: int, rank: int = 0) -> None:
    process_seed = int(seed) + int(rank) * 1000
    random.seed(process_seed)
    np.random.seed(process_seed)
    torch.manual_seed(process_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(process_seed)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


# =============================================================================
# LABEL / DATASET VALIDATION
# =============================================================================
def reorder_corners(pts: np.ndarray) -> np.ndarray:
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    center = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    return pts[np.argsort(angles)].astype(np.float32)


def polygon_area_np(pts: np.ndarray) -> float:
    pts = np.asarray(pts, dtype=np.float64)
    x = pts[:, 0]
    y = pts[:, 1]
    return 0.5 * abs(float(np.sum(x * np.roll(y, -1) - y * np.roll(x, -1))))


def _quad_is_convex(pts: np.ndarray, eps: float = 1e-8) -> bool:
    pts = np.asarray(pts, dtype=np.float64).reshape(4, 2)
    signs = []
    for i in range(4):
        a = pts[(i + 1) % 4] - pts[i]
        b = pts[(i + 2) % 4] - pts[(i + 1) % 4]
        cross = a[0] * b[1] - a[1] * b[0]
        if abs(cross) > eps:
            signs.append(np.sign(cross))
    return len(signs) == 4 and (all(s > 0 for s in signs) or all(s < 0 for s in signs))


def validate_quad(pts: np.ndarray) -> Tuple[bool, str]:
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    if not np.isfinite(pts).all():
        return False, "coordonnée NaN/Inf"
    if (pts < 0.0).any() or (pts > 1.0).any():
        return False, "coordonnée hors [0,1]"
    pts = reorder_corners(pts)
    if polygon_area_np(pts) < 5e-5:
        return False, "aire trop petite/dégénérée"
    edges = np.linalg.norm(np.roll(pts, -1, axis=0) - pts, axis=1)
    if float(edges.min()) < 0.005:
        return False, "arête trop courte"
    if not _quad_is_convex(pts):
        return False, "quadrilatère non convexe / ordre incohérent"
    return True, "ok"


def read_label(label_path: Path, strict: bool = True):
    objects = []
    errors = []
    with open(label_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 9:
                errors.append(f"ligne {line_no}: 9 valeurs attendues, {len(parts)} reçues")
                continue
            try:
                cls = int(float(parts[0]))
                coords = np.asarray([float(v) for v in parts[1:]], dtype=np.float32)
            except ValueError:
                errors.append(f"ligne {line_no}: valeur non numérique")
                continue
            if cls != 0:
                errors.append(f"ligne {line_no}: classe {cls} non supportée (attendu 0)")
                continue
            pts = reorder_corners(coords.reshape(4, 2))
            ok, reason = validate_quad(pts)
            if not ok:
                errors.append(f"ligne {line_no}: {reason}")
                continue
            objects.append({"cls": 0, "corners": pts})

    if strict and errors:
        raise ValueError(f"{label_path}: " + " ; ".join(errors[:5]))
    if strict and not objects:
        raise ValueError(f"{label_path}: aucun objet valide")
    return objects


def resolve_label_path(img_path: Path) -> Path | None:
    direct = LABELS_DIR / f"{img_path.stem}.txt"
    if direct.exists():
        return direct
    m = re.fullmatch(r"image(\d+)", img_path.stem, flags=re.IGNORECASE)
    if m:
        alt = LABELS_DIR / f"label{m.group(1)}.txt"
        if alt.exists():
            return alt
    return None


def collect_pairs():
    if not IMAGES_DIR.exists() or not LABELS_DIR.exists():
        raise RuntimeError(f"Dataset absent : {IMAGES_DIR} / {LABELS_DIR}")

    pairs = []
    problems = []
    candidates = sorted(p for p in IMAGES_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    for img_path in tqdm(candidates, desc="Validation dataset", disable=not IS_MAIN, dynamic_ncols=True):
        label_path = resolve_label_path(img_path)
        if label_path is None:
            problems.append(f"{img_path.name}: label introuvable")
            continue
        try:
            with Image.open(img_path) as im:
                im.verify()
            read_label(label_path, strict=True)
        except Exception as exc:
            problems.append(f"{img_path.name}: {exc}")
            continue
        pairs.append((img_path, label_path))

    if problems:
        preview = "\n".join(f"  - {x}" for x in problems[:30])
        raise RuntimeError(
            f"Dataset refusé : {len(problems)} problème(s) détecté(s).\n{preview}\n"
            "Corrige ces fichiers avant de lancer l'entraînement."
        )
    if not pairs:
        raise RuntimeError("Aucune paire image/label valide.")
    rank0_print(f"✅ Dataset validé : {len(pairs)} paires propres")
    return pairs


def dataset_signature(pairs) -> str:
    h = hashlib.sha256()
    for img_path, label_path in sorted(pairs, key=lambda p: p[0].name):
        h.update(img_path.name.encode("utf-8"))
        h.update(str(img_path.stat().st_size).encode("ascii"))
        h.update(label_path.name.encode("utf-8"))
        h.update(label_path.read_bytes())
    return h.hexdigest()


def _class_signature(label_path: Path) -> Tuple[int, ...]:
    return (len(read_label(label_path, strict=True)),)


def _write_split_manifest(train_p, val_p, test_p, signature: str) -> None:
    if not IS_MAIN:
        return
    payload = {
        "version": 2,
        "seed": SEED,
        "dataset_dir": str(DATA_DIR),
        "dataset_signature": signature,
        "train": [p[0].name for p in train_p],
        "val": [p[0].name for p in val_p],
        "test": [p[0].name for p in test_p],
    }
    tmp = SPLIT_MANIFEST_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, SPLIT_MANIFEST_PATH)


def _try_load_split_manifest(pairs, signature: str):
    if not SPLIT_MANIFEST_PATH.exists():
        return None
    try:
        payload = json.loads(SPLIT_MANIFEST_PATH.read_text(encoding="utf-8"))
    except Exception:
        return None
    if payload.get("version") != 2 or payload.get("dataset_signature") != signature:
        rank0_print("♻️ Dataset modifié : ancien split ignoré et recréé.")
        return None

    by_name = {p[0].name: p for p in pairs}
    train_p = [by_name[n] for n in payload.get("train", []) if n in by_name]
    val_p = [by_name[n] for n in payload.get("val", []) if n in by_name]
    test_p = [by_name[n] for n in payload.get("test", []) if n in by_name]
    used = {p[0].name for p in train_p + val_p + test_p}
    if used != set(by_name) or not val_p or not test_p:
        return None
    return train_p, val_p, test_p


def split_pairs(pairs):
    signature = dataset_signature(pairs)
    existing = _try_load_split_manifest(pairs, signature)
    if existing is not None:
        rank0_print("✅ Split V6 existant réutilisé (signature dataset identique)")
        return existing

    groups = defaultdict(list)
    for pair in pairs:
        groups[_class_signature(pair[1])].append(pair)

    rng = random.Random(SEED)
    train_p, val_p, test_p = [], [], []
    for signature_group in sorted(groups, key=str):
        group = sorted(groups[signature_group], key=lambda p: p[0].name)
        rng.shuffle(group)
        n = len(group)
        n_val = int(round(n * VAL_RATIO))
        n_test = int(round(n * TEST_RATIO))
        if n >= 10:
            n_val = max(1, n_val)
            n_test = max(1, n_test)
        while n_val + n_test >= n and (n_val > 0 or n_test > 0):
            if n_test >= n_val and n_test > 0:
                n_test -= 1
            elif n_val > 0:
                n_val -= 1
        val_p.extend(group[:n_val])
        test_p.extend(group[n_val:n_val + n_test])
        train_p.extend(group[n_val + n_test:])

    rng.shuffle(train_p)
    rng.shuffle(val_p)
    rng.shuffle(test_p)
    _write_split_manifest(train_p, val_p, test_p, signature)
    barrier()
    return train_p, val_p, test_p


# =============================================================================
# PREPROCESS / AUGMENTATION
# =============================================================================
def letterbox_resize(image: Image.Image, target_size: int, objects):
    w, h = image.size
    scale = min(target_size / w, target_size / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    scale_x = new_w / w
    scale_y = new_h / h
    left = (target_size - new_w) // 2
    top = (target_size - new_h) // 2

    resampling = getattr(Image, "Resampling", Image)
    resized = image.resize((new_w, new_h), resample=resampling.BILINEAR)
    canvas = Image.new("RGB", (target_size, target_size), (114, 114, 114))
    canvas.paste(resized, (left, top))

    transformed = []
    for obj in objects:
        pts = np.asarray(obj["corners"], dtype=np.float32).copy()
        px = pts[:, 0] * w * scale_x + left
        py = pts[:, 1] * h * scale_y + top
        out = np.stack([px / target_size, py / target_size], axis=1)
        transformed.append({"cls": int(obj["cls"]), "corners": reorder_corners(out)})
    return canvas, transformed


class GeometricAug:
    def __init__(self, hflip: float = 0.5, vflip: float = 0.0, rot_deg: float = ROT_MAX_DEG):
        self.hflip = hflip
        self.vflip = vflip
        self.rot_deg = rot_deg

    @staticmethod
    def _rotate_points_pixel(pts_norm: np.ndarray, w: int, h: int, angle_deg: float) -> np.ndarray:
        pts = np.asarray(pts_norm, dtype=np.float64).copy()
        x = pts[:, 0] * w
        y = pts[:, 1] * h
        cx, cy = w / 2.0, h / 2.0
        dx, dy = x - cx, y - cy
        a = math.radians(angle_deg)
        ca, sa = math.cos(a), math.sin(a)
        xr = ca * dx + sa * dy + cx
        yr = -sa * dx + ca * dy + cy
        return np.stack([xr / w, yr / h], axis=1).astype(np.float32)

    @staticmethod
    def _inside(pts: np.ndarray) -> bool:
        return bool(np.isfinite(pts).all() and (pts >= 0.0).all() and (pts <= 1.0).all())

    def __call__(self, image: Image.Image, objects):
        w, h = image.size
        out_img = image
        out_objs = [{"cls": int(o["cls"]), "corners": np.asarray(o["corners"], dtype=np.float32).copy()} for o in objects]

        if random.random() < self.hflip:
            out_img = out_img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            for obj in out_objs:
                obj["corners"][:, 0] = 1.0 - obj["corners"][:, 0]
                obj["corners"] = reorder_corners(obj["corners"])

        if self.vflip > 0.0 and random.random() < self.vflip:
            out_img = out_img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
            for obj in out_objs:
                obj["corners"][:, 1] = 1.0 - obj["corners"][:, 1]
                obj["corners"] = reorder_corners(obj["corners"])

        angle = random.uniform(-self.rot_deg, self.rot_deg)
        if abs(angle) < 0.5:
            return out_img, out_objs

        rotated_objs = []
        for obj in out_objs:
            pts = self._rotate_points_pixel(obj["corners"], w, h, angle)
            if not self._inside(pts):
                return out_img, out_objs
            pts = reorder_corners(pts)
            ok, _ = validate_quad(pts)
            if not ok:
                return out_img, out_objs
            rotated_objs.append({"cls": int(obj["cls"]), "corners": pts})

        resampling = getattr(Image, "Resampling", Image)
        rotated_img = out_img.rotate(
            angle,
            resample=resampling.BILINEAR,
            expand=False,
            fillcolor=(114, 114, 114),
        )
        return rotated_img, rotated_objs


class PlankDataset(Dataset):
    def __init__(self, pairs, augment: bool = False):
        self.pairs = list(pairs)
        self.geo_aug = GeometricAug() if augment else None
        self.photo_tf = (
            transforms.Compose(
                [
                    transforms.RandomApply(
                        [transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.035)],
                        p=0.80,
                    ),
                    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.15, 1.0))], p=0.12),
                ]
            )
            if augment
            else None
        )
        self.to_tensor = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ]
        )

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, label_path = self.pairs[idx]
        with Image.open(img_path) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
        objects = read_label(label_path, strict=True)
        if self.geo_aug is not None:
            image, objects = self.geo_aug(image, objects)
        if self.photo_tf is not None:
            image = self.photo_tf(image)
        image, objects = letterbox_resize(image, IMG_SIZE, objects)
        return self.to_tensor(image), objects


def collate_fn(batch):
    return torch.stack([x[0] for x in batch]), [x[1] for x in batch]


def make_loaders(train_p, val_p, test_p):
    cpu = os.cpu_count() or 4
    workers = max(1, min(4, cpu // max(WORLD_SIZE, 1)))
    common = {
        "num_workers": workers,
        "pin_memory": DEVICE_TYPE == "cuda",
        "persistent_workers": False,
        "collate_fn": collate_fn,
        "worker_init_fn": seed_worker,
        "prefetch_factor": 2,
    }
    train_ds = PlankDataset(train_p, augment=True)
    val_ds = PlankDataset(val_p, augment=False)
    test_ds = PlankDataset(test_p, augment=False)

    train_sampler = (
        DistributedSampler(train_ds, num_replicas=WORLD_SIZE, rank=RANK, shuffle=True, seed=SEED)
        if IS_DISTRIBUTED
        else None
    )
    generator = torch.Generator().manual_seed(SEED + RANK)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=train_sampler is None,
        sampler=train_sampler,
        generator=generator if train_sampler is None else None,
        **common,
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * VAL_BATCH_MULT, shuffle=False, **common)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * VAL_BATCH_MULT, shuffle=False, **common)
    return train_loader, val_loader, test_loader, train_sampler


# =============================================================================
# EMA / FREEZING / OPTIMIZER
# =============================================================================
class ModelEMA:
    def __init__(self, model: nn.Module, max_decay: float = EMA_MAX_DECAY, tau: float = EMA_TAU):
        self.ema = copy.deepcopy(unwrap_model(model)).eval()
        self.max_decay = float(max_decay)
        self.tau = float(tau)
        self.updates = 0
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def current_decay(self) -> float:
        # Starts near 0 and asymptotically approaches max_decay.
        return self.max_decay * (1.0 - math.exp(-self.updates / self.tau))

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        self.updates += 1
        d = self.current_decay()
        model_state = unwrap_model(model).state_dict()
        for key, value in self.ema.state_dict().items():
            src = model_state[key].detach()
            if value.dtype.is_floating_point:
                value.mul_(d).add_(src, alpha=1.0 - d)
            else:
                value.copy_(src)


def set_backbone_trainable(model: nn.Module, epoch: int) -> str:
    backbone = unwrap_model(model).backbone
    for p in backbone.parameters():
        p.requires_grad_(False)

    if epoch >= UNFREEZE_LAST_EPOCH:
        for p in backbone.layer3.parameters():
            p.requires_grad_(True)
        for p in backbone.layer4.parameters():
            p.requires_grad_(True)
    if epoch >= UNFREEZE_ALL_EPOCH:
        for p in backbone.parameters():
            p.requires_grad_(True)

    if epoch < UNFREEZE_LAST_EPOCH:
        phase = "GELÉ"
    elif epoch < UNFREEZE_ALL_EPOCH:
        phase = "PARTIEL layer3+4"
    else:
        phase = "COMPLET"
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    total = sum(p.numel() for p in backbone.parameters())
    rank0_print(f"🔓 Backbone epoch {epoch}: {phase} | {trainable:,}/{total:,} paramètres entraînables")
    return phase


def build_optimizer(model: nn.Module):
    # Include ALL parameters from the start. Frozen parameters simply have no
    # gradients; this lets us preserve Adam moments when unfreezing later.
    base = unwrap_model(model)
    backbone_params = [p for n, p in base.named_parameters() if n.startswith("backbone.")]
    head_params = [p for n, p in base.named_parameters() if not n.startswith("backbone.")]
    return torch.optim.AdamW(
        [
            {"params": head_params, "lr": LR_HEAD, "base_lr": LR_HEAD, "group_name": "head"},
            {"params": backbone_params, "lr": LR_BACKBONE, "base_lr": LR_BACKBONE, "group_name": "backbone"},
        ],
        weight_decay=WEIGHT_DECAY,
    )


def set_cosine_lr(optimizer, epoch: int) -> None:
    progress = (epoch - 1) / max(EPOCHS - 1, 1)
    factor = MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * 0.5 * (1.0 + math.cos(math.pi * progress))
    for group in optimizer.param_groups:
        group["lr"] = float(group["base_lr"]) * factor


# =============================================================================
# DETECTION METRICS
# =============================================================================
def _match_image(preds: List[dict], gts: List[np.ndarray], iou_thresh: float, conf: float):
    preds = [p for p in preds if p["score"] >= conf]
    preds = sorted(preds, key=lambda p: p["score"], reverse=True)
    matched = set()
    tp = 0
    fp = 0
    matched_ious = []
    for pred in preds:
        best_iou = 0.0
        best_j = -1
        for j, gt in enumerate(gts):
            if j in matched:
                continue
            iou = quad_iou_np(pred["corners"], gt)
            if iou > best_iou:
                best_iou = iou
                best_j = j
        if best_j >= 0 and best_iou >= iou_thresh:
            matched.add(best_j)
            tp += 1
            matched_ious.append(best_iou)
        else:
            fp += 1
    fn = len(gts) - len(matched)
    return tp, fp, fn, matched_ious


def _average_precision(all_preds: List[dict], gt_by_image: Dict[int, List[np.ndarray]], iou_thresh: float) -> float:
    total_gt = sum(len(v) for v in gt_by_image.values())
    if total_gt == 0:
        return 0.0
    preds = sorted(all_preds, key=lambda p: p["score"], reverse=True)
    matched_by_image = {k: set() for k in gt_by_image}
    tps, fps = [], []

    for pred in preds:
        image_id = pred["image_id"]
        gts = gt_by_image.get(image_id, [])
        matched = matched_by_image.setdefault(image_id, set())
        best_iou = 0.0
        best_j = -1
        for j, gt in enumerate(gts):
            if j in matched:
                continue
            iou = quad_iou_np(pred["corners"], gt)
            if iou > best_iou:
                best_iou = iou
                best_j = j
        is_tp = best_j >= 0 and best_iou >= iou_thresh
        if is_tp:
            matched.add(best_j)
        tps.append(1.0 if is_tp else 0.0)
        fps.append(0.0 if is_tp else 1.0)

    if not tps:
        return 0.0
    tp_cum = np.cumsum(tps)
    fp_cum = np.cumsum(fps)
    recall = tp_cum / max(total_gt, 1)
    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)

    mrec = np.concatenate([[0.0], recall, [1.0]])
    mpre = np.concatenate([[0.0], precision, [0.0]])
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def detection_summary(all_preds: List[dict], gt_by_image: Dict[int, List[np.ndarray]]) -> Dict[str, float]:
    ap50 = _average_precision(all_preds, gt_by_image, 0.50)
    ap75 = _average_precision(all_preds, gt_by_image, 0.75)

    tp = fp = fn = 0
    ious = []
    by_image_preds = defaultdict(list)
    for p in all_preds:
        by_image_preds[p["image_id"]].append(p)
    for image_id, gts in gt_by_image.items():
        a, b, c, d = _match_image(by_image_preds.get(image_id, []), gts, 0.50, VAL_REPORT_CONF)
        tp += a
        fp += b
        fn += c
        ious.extend(d)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2.0 * precision * recall / (precision + recall) if precision + recall else 0.0
    mean_iou = float(np.mean(ious)) if ious else 0.0
    quality = 0.70 * ap50 + 0.30 * ap75
    return {
        "ap50": ap50,
        "ap75": ap75,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_iou": mean_iou,
        "quality": quality,
        "tp": float(tp),
        "fp": float(fp),
        "fn": float(fn),
    }


# =============================================================================
# EPOCH LOOP
# =============================================================================
def _set_frozen_batchnorm_modes(model: nn.Module, epoch: int) -> None:
    backbone = unwrap_model(model).backbone
    if epoch < UNFREEZE_LAST_EPOCH:
        backbone.eval()
    elif epoch < UNFREEZE_ALL_EPOCH:
        backbone.stem.eval()
        backbone.layer1.eval()
        backbone.layer2.eval()
        backbone.layer3.train()
        backbone.layer4.train()
    else:
        backbone.train()


def run_train_epoch(model, loader, optimizer, scaler, epoch: int, ema: ModelEMA):
    model.train()
    _set_frozen_batchnorm_modes(model, epoch)
    optimizer.zero_grad(set_to_none=True)

    sums = defaultdict(float)
    n_images = 0
    n_batches = len(loader)
    bar = tqdm(loader, desc=f"TRAIN {epoch}/{EPOCHS}", disable=not IS_MAIN, dynamic_ncols=True)

    for step, (images, batch_objects) in enumerate(bar):
        images = images.to(DEVICE, non_blocking=True)
        bs = images.shape[0]

        group_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_len = min(GRAD_ACCUM_STEPS, n_batches - group_start)
        should_step = ((step + 1) % GRAD_ACCUM_STEPS == 0) or (step + 1 == n_batches)
        sync_ctx = nullcontext() if should_step or not isinstance(model, DDP) else model.no_sync()

        with sync_ctx:
            with torch.amp.autocast(
                device_type=DEVICE_TYPE,
                dtype=torch.float16 if DEVICE_TYPE == "cuda" else torch.bfloat16,
                enabled=AMP_ENABLED,
            ):
                outputs = model(images)
                loss, breakdown = combined_loss_v6(outputs, batch_objects, DEVICE)
                scaled_loss = loss / float(group_len)

            if scaler.is_enabled():
                scaler.scale(scaled_loss).backward()
            else:
                scaled_loss.backward()

        if should_step:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad and p.grad is not None],
                GRAD_CLIP_NORM,
            )
            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            ema.update(model)

        sums["loss"] += float(loss.detach()) * bs
        for key, value in breakdown.items():
            sums[key] += float(value) * bs
        n_images += bs
        if IS_MAIN:
            bar.set_postfix(loss=f"{sums['loss'] / max(n_images, 1):.4f}")

    keys = ["loss", "total", "heatmap", "corner", "offset", "geometry"]
    vals = torch.tensor([sums[k] for k in keys] + [float(n_images)], device=DEVICE, dtype=torch.float64)
    if IS_DISTRIBUTED:
        dist.all_reduce(vals, op=dist.ReduceOp.SUM)
    denom = max(float(vals[-1].item()), 1.0)
    return {k: float(vals[i].item()) / denom for i, k in enumerate(keys)}


@torch.no_grad()
def run_eval_epoch(model: nn.Module, loader, epoch: int | None, phase: str = "val"):
    model.eval()
    sums = defaultdict(float)
    n_images = 0
    all_preds: List[dict] = []
    gt_by_image: Dict[int, List[np.ndarray]] = {}
    next_image_id = 0

    desc = f"{phase.upper()} {epoch}/{EPOCHS}" if epoch is not None else phase.upper()
    bar = tqdm(loader, desc=desc, disable=not IS_MAIN, dynamic_ncols=True)
    for images, batch_objects in bar:
        images = images.to(DEVICE, non_blocking=True)
        bs = images.shape[0]
        with torch.amp.autocast(
            device_type=DEVICE_TYPE,
            dtype=torch.float16 if DEVICE_TYPE == "cuda" else torch.bfloat16,
            enabled=AMP_ENABLED,
        ):
            outputs = model(images)
            loss, breakdown = combined_loss_v6(outputs, batch_objects, DEVICE)

        det_batch = decode_predictions(
            *outputs,
            score_thresh=VAL_DECODE_CONF,
            topk=VAL_TOPK,
            nms_iou=VAL_NMS_IOU,
        )

        for bi in range(bs):
            image_id = next_image_id
            next_image_id += 1
            gt_by_image[image_id] = [np.asarray(o["corners"], dtype=np.float64) for o in batch_objects[bi]]
            for det in det_batch[bi]:
                all_preds.append(
                    {
                        "image_id": image_id,
                        "score": float(det["score"]),
                        "corners": np.asarray(det["corners"], dtype=np.float64),
                    }
                )

        sums["loss"] += float(loss.detach()) * bs
        for key, value in breakdown.items():
            sums[key] += float(value) * bs
        n_images += bs
        bar.set_postfix(loss=f"{sums['loss'] / max(n_images, 1):.4f}")

    stats = {k: v / max(n_images, 1) for k, v in sums.items()}
    stats.update(detection_summary(all_preds, gt_by_image))
    return stats


# =============================================================================
# CHECKPOINTS / PLOTS / PERSISTENCE
# =============================================================================
def save_ckpt(path: Path, model, optimizer, scaler, ema, epoch, histories, best_quality, best_val_loss, patience, val_stats):
    if not IS_MAIN:
        return
    payload = {
        "epoch": int(epoch),
        "model": unwrap_model(model).state_dict(),
        "ema": ema.ema.state_dict(),
        "ema_updates": int(ema.updates),
        "optim": optimizer.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "histories": histories,
        "best_quality": float(best_quality),
        "best_metric_name": "0.70*AP50 + 0.30*AP75",
        "best_val_loss": float(best_val_loss),
        "epochs_without_improvement": int(patience),
        "last_val_metrics": {k: float(v) for k, v in val_stats.items() if isinstance(v, (int, float))},
        "config": {
            "model_version": MODEL_VERSION,
            "run_id": RUN_ID,
            "data_dir": str(DATA_DIR),
            "img_size": IMG_SIZE,
            "batch_size_per_gpu": BATCH_SIZE,
            "world_size": WORLD_SIZE,
            "grad_accum": GRAD_ACCUM_STEPS,
            "lr_head": LR_HEAD,
            "lr_backbone": LR_BACKBONE,
            "unfreeze_last_epoch": UNFREEZE_LAST_EPOCH,
            "unfreeze_all_epoch": UNFREEZE_ALL_EPOCH,
            "ema_max_decay": EMA_MAX_DECAY,
            "ema_tau": EMA_TAU,
        },
    }
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def load_ckpt(model, optimizer, scaler, ema, path: Path):
    empty_hist = {"train_loss": [], "val_loss": [], "ap50": [], "ap75": [], "quality": []}
    if not path.exists():
        return 1, empty_hist, float("-inf"), float("inf"), 0
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    version = ckpt.get("config", {}).get("model_version")
    if version != MODEL_VERSION:
        raise RuntimeError(
            f"Checkpoint incompatible: model_version={version!r}, attendu {MODEL_VERSION!r}. "
            "Checkpoint d’une autre version détecté. Pour V6, repars avec un nouveau checkpoint V6."
        )
    unwrap_model(model).load_state_dict(ckpt["model"], strict=True)
    optimizer.load_state_dict(ckpt["optim"])
    if scaler is not None and ckpt.get("scaler"):
        scaler.load_state_dict(ckpt["scaler"])
    if ckpt.get("ema"):
        ema.ema.load_state_dict(ckpt["ema"], strict=True)
        ema.updates = int(ckpt.get("ema_updates", 0))
    return (
        int(ckpt.get("epoch", 0)) + 1,
        ckpt.get("histories", empty_hist),
        float(ckpt.get("best_quality", float("-inf"))),
        float(ckpt.get("best_val_loss", float("inf"))),
        int(ckpt.get("epochs_without_improvement", 0)),
    )


def save_training_plot(histories):
    if not IS_MAIN or not histories.get("train_loss"):
        return
    epochs = np.arange(1, len(histories["train_loss"]) + 1)

    fig = plt.figure(figsize=(11, 6))
    ax1 = fig.add_subplot(111)
    ax1.plot(epochs, histories["train_loss"], label="train loss")
    ax1.plot(epochs, histories["val_loss"], label="val loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(alpha=0.25)
    ax1.legend(loc="upper left")

    ax2 = ax1.twinx()
    ax2.plot(epochs, histories["ap50"], linestyle="--", label="AP50")
    ax2.plot(epochs, histories["ap75"], linestyle="--", label="AP75")
    ax2.set_ylabel("AP")
    ax2.set_ylim(0.0, 1.0)
    ax2.legend(loc="upper right")

    plt.title("PlankEye V6 — losses + detection AP")
    fig.tight_layout()
    fig.savefig(PLOT_PATH, dpi=150)
    plt.close(fig)


def reset_training_state_if_requested():
    if IS_MAIN and START_FROM_SCRATCH:
        rank0_print("🧹 START_FROM_SCRATCH=True : suppression de l'état V6")
        for p in [BEST_MODEL_PATH, LAST_CHECKPOINT_PATH, SPLIT_MANIFEST_PATH, PLOT_PATH]:
            if p.exists():
                p.unlink()
    barrier()


def persist_checkpoints_to_kaggle(epoch: int) -> bool:
    if not IS_MAIN or not KAGGLE_DATASET_ID:
        return False
    kaggle_exe = shutil.which("kaggle")
    if kaggle_exe is None:
        rank0_print("⚠️ Persistence Kaggle ignorée : commande kaggle absente.")
        return False
    try:
        PERSIST_DIR.mkdir(parents=True, exist_ok=True)
        for source in [BEST_MODEL_PATH, LAST_CHECKPOINT_PATH, PLOT_PATH, SPLIT_MANIFEST_PATH]:
            if source.exists():
                shutil.copy2(source, PERSIST_DIR / source.name)
        metadata = {
            "title": "chekpoints-backbone18",
            "id": KAGGLE_DATASET_ID,
            "licenses": [{"name": "other"}],
            "isPrivate": True,
        }
        (PERSIST_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
        result = subprocess.run(
            [kaggle_exe, "datasets", "version", "-p", str(PERSIST_DIR), "-m", f"PlankEye V6 epoch {epoch}"],
            text=True,
            capture_output=True,
            check=False,
        )
        if result.returncode != 0:
            rank0_print("⚠️ Persistence Kaggle échouée:", result.stderr.strip())
            return False
        rank0_print(f"💾 Checkpoints V6 persistés sur {KAGGLE_DATASET_ID} (epoch {epoch})")
        return True
    except Exception as exc:
        rank0_print(f"⚠️ Persistence Kaggle échouée: {exc}")
        return False


# =============================================================================
# MAIN
# =============================================================================
def main():
    setup_distributed()
    try:
        set_seed(SEED, RANK)
        reset_training_state_if_requested()

        if IS_MAIN:
            print("=" * 78)
            print("🚀 PLANKEYE V6 — ABSOLUTE CORNERS + AP CHECKPOINTING")
            print("=" * 78)
            print(f"Model version            : {MODEL_VERSION}")
            print(f"Dataset                  : {DATA_DIR}")
            print(f"GPU utilisés             : {WORLD_SIZE}")
            print(f"Batch / GPU              : {BATCH_SIZE}")
            print(f"Gradient accumulation    : {GRAD_ACCUM_STEPS}")
            print(f"Batch effectif approx.   : {BATCH_SIZE * WORLD_SIZE * GRAD_ACCUM_STEPS}")
            print(f"Epochs max               : {EPOCHS}")
            print(f"Dégel layer3+4           : epoch {UNFREEZE_LAST_EPOCH}")
            print(f"Dégel complet            : epoch {UNFREEZE_ALL_EPOCH}")
            print(f"Best metric              : 0.70*AP50 + 0.30*AP75")
            print("=" * 78)

        pairs = collect_pairs()
        train_p, val_p, test_p = split_pairs(pairs)
        barrier()
        train_loader, val_loader, test_loader, train_sampler = make_loaders(train_p, val_p, test_p)
        rank0_print(f"📊 Split : train={len(train_p)} | val={len(val_p)} | test={len(test_p)}")

        raw_model = PlankEyeV6(pretrained_backbone=PRETRAINED_BACKBONE).to(DEVICE)
        optimizer = build_optimizer(raw_model)
        scaler = torch.amp.GradScaler(DEVICE_TYPE, enabled=AMP_ENABLED)
        ema = ModelEMA(raw_model)

        # Set phase before wrapping with DDP.
        resume_epoch = 1
        if LAST_CHECKPOINT_PATH.exists():
            preview = torch.load(LAST_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
            resume_epoch = int(preview.get("epoch", 0)) + 1
        set_backbone_trainable(raw_model, resume_epoch)

        start_epoch, histories, best_quality, best_val_loss, patience = load_ckpt(
            raw_model, optimizer, scaler, ema, LAST_CHECKPOINT_PATH
        )
        set_backbone_trainable(raw_model, start_epoch)
        train_model = wrap_for_training(raw_model)

        if IS_MAIN:
            total_params = sum(p.numel() for p in raw_model.parameters())
            rank0_print(f"🧠 Paramètres : {total_params:,}")
            rank0_print(f"🔄 Départ epoch {start_epoch} | best quality={best_quality:.4f} | patience={patience}")

        for epoch in range(start_epoch, EPOCHS + 1):
            if train_sampler is not None:
                train_sampler.set_epoch(epoch)

            # Rebuild ONLY DDP wrapper when the set of trainable parameters changes.
            # Optimizer is kept, so Adam moments are not lost.
            if epoch in (UNFREEZE_LAST_EPOCH, UNFREEZE_ALL_EPOCH) and epoch != start_epoch:
                barrier()
                if IS_DISTRIBUTED:
                    del train_model
                set_backbone_trainable(raw_model, epoch)
                train_model = wrap_for_training(raw_model)
                if DEVICE_TYPE == "cuda":
                    torch.cuda.empty_cache()
                barrier()
            else:
                set_backbone_trainable(raw_model, epoch)

            set_cosine_lr(optimizer, epoch)
            train_stats = run_train_epoch(train_model, train_loader, optimizer, scaler, epoch, ema)
            barrier()

            stop_training = False
            if IS_MAIN:
                # Validate the exact EMA model that will be preferred at inference.
                val_stats = run_eval_epoch(ema.ema, val_loader, epoch, phase="val")
                val_loss = float(val_stats.get("loss", float("inf")))
                quality = float(val_stats["quality"])

                histories.setdefault("train_loss", []).append(float(train_stats["loss"]))
                histories.setdefault("val_loss", []).append(val_loss)
                histories.setdefault("ap50", []).append(float(val_stats["ap50"]))
                histories.setdefault("ap75", []).append(float(val_stats["ap75"]))
                histories.setdefault("quality", []).append(quality)

                print(
                    f"📊 Epoch {epoch:03d}/{EPOCHS} | "
                    f"train={train_stats['loss']:.4f} val={val_loss:.4f} | "
                    f"AP50={val_stats['ap50']:.3f} AP75={val_stats['ap75']:.3f} | "
                    f"P={val_stats['precision']:.3f} R={val_stats['recall']:.3f} "
                    f"IoU={val_stats['mean_iou']:.3f} | Q={quality:.4f} | "
                    f"EMA_d={ema.current_decay():.5f}"
                )

                improved = quality > best_quality + EARLY_STOP_MIN_DELTA
                if improved:
                    best_quality = quality
                    best_val_loss = val_loss
                    patience = 0
                    save_ckpt(
                        BEST_MODEL_PATH, raw_model, optimizer, scaler, ema, epoch,
                        histories, best_quality, best_val_loss, patience, val_stats,
                    )
                    print(f"⭐ Nouveau BEST epoch {epoch}: Q={quality:.4f} AP50={val_stats['ap50']:.3f}")
                else:
                    patience += 1
                    print(f"⏳ Pas d'amélioration detection : {patience}/{EARLY_STOP_PATIENCE}")

                save_ckpt(
                    LAST_CHECKPOINT_PATH, raw_model, optimizer, scaler, ema, epoch,
                    histories, best_quality, best_val_loss, patience, val_stats,
                )
                save_training_plot(histories)
                if epoch % PERSIST_EVERY == 0:
                    persist_checkpoints_to_kaggle(epoch)

                stop_training = patience >= EARLY_STOP_PATIENCE
                if stop_training:
                    print(f"🛑 Early stopping. Best quality={best_quality:.4f}")

            stop_training = broadcast_stop(stop_training)
            barrier()
            if stop_training:
                break

        barrier()
        if IS_MAIN and BEST_MODEL_PATH.exists():
            best = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
            ema.ema.load_state_dict(best["ema"], strict=True)
            test_stats = run_eval_epoch(ema.ema, test_loader, None, phase="test")
            print("=" * 78)
            print("🧪 TEST FINAL — meilleur checkpoint EMA")
            print(f"AP50      : {test_stats['ap50']:.4f}")
            print(f"AP75      : {test_stats['ap75']:.4f}")
            print(f"Precision : {test_stats['precision']:.4f}")
            print(f"Recall    : {test_stats['recall']:.4f}")
            print(f"Mean IoU  : {test_stats['mean_iou']:.4f}")
            print(f"Loss      : {test_stats['loss']:.6f}")
            print(f"Best ckpt : {BEST_MODEL_PATH}")
            print("=" * 78)
        barrier()
    finally:
        cleanup_distributed()


if __name__ == "__main__":
    main()


## 6 — Vérifier la syntaxe et les réglages critiques


In [ ]:
from pathlib import Path

TRAIN_SCRIPT = Path(
    "/kaggle/working/PlankEyev2_multipieces/train_plankeye_ddp.py"
)

source = TRAIN_SCRIPT.read_text(encoding="utf-8")
compile(source, str(TRAIN_SCRIPT), "exec")

checks = {
    "BATCH_SIZE = 16": "BATCH_SIZE = 16" in source,
    "find_unused_parameters=False": "find_unused_parameters=False" in source,
    "FileStore": 'init_method=f"file://{store_path}"' in source,
    "PERSIST_EVERY = 5": "PERSIST_EVERY = 5" in source,
    "CLI directe kaggle": 'shutil.which("kaggle")' in source,
}

print("Syntaxe OK :", TRAIN_SCRIPT)
print("Nombre de lignes :", len(source.splitlines()))
print()

for name, ok in checks.items():
    print(f"{name:<34} -> {'OK' if ok else 'MANQUANT'}")

if not all(checks.values()):
    raise RuntimeError("Un réglage critique manque dans le script.")

## mettre batch size 12 apres epoch 8 puis 8 apres epoch 20

## 7 — Vérifier le mode de démarrage


In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")

RUN_ID = "v5_run"

LAST = PROJECT / "last_checkpoint_v6.pt"
BEST = PROJECT / "best_checkpoint_v6.pt"

print("=" * 72)
print("MODE DE DÉMARRAGE")
print("=" * 72)

if LAST.exists():
    ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
    config = ckpt.get("config") or {}
    compatible = (config.get("run_id") == RUN_ID and config.get("pretrained_backbone") is True)
else:
    ckpt = None
    compatible = False

if compatible:
    epoch = int(ckpt.get("epoch", 0))
    print("✅ REPRISE")
    print("Dernier epoch :", epoch)
    print("Prochain epoch:", epoch + 1)
else:
    print("✅ PREMIER LANCEMENT")
    print("Départ        : epoch 1")

print()
print("LAST :", LAST)
print("BEST :", BEST)
## 8 — Lancer l'entraînement sur **2 GPU**


In [ ]:
import os
import torch

LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))

DEVICE = torch.device(
    f"cuda:{LOCAL_RANK}" if torch.cuda.is_available() else "cpu"
)

raw_model = PlankEyeV5().to(DEVICE)


def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


# ─────────────────────────────────────────────
# TOTAL
# ─────────────────────────────────────────────
total, trainable = count_params(raw_model)

print("=" * 65)
print("📊 PARAMÈTRES PLANKEYE V5")
print("=" * 65)

print(f"Device                  : {DEVICE}")
print(f"Paramètres totaux       : {total:,}")
print(f"Paramètres entraînables : {trainable:,}")
print(f"Paramètres gelés        : {total - trainable:,}")

print()
print("-" * 65)
print("📦 PARAMÈTRES PAR COMPOSANT")
print("-" * 65)


# ─────────────────────────────────────────────
# BACKBONE
# ─────────────────────────────────────────────
if hasattr(raw_model, "backbone"):
    backbone_total, backbone_trainable = count_params(raw_model.backbone)

    print(
        f"Backbone                : "
        f"{backbone_total:,} "
        f"(trainables: {backbone_trainable:,}) "
        f"[{100 * backbone_total / total:.2f}%]"
    )


# ─────────────────────────────────────────────
# RESTE DU MODÈLE = HEAD / NECK
# ─────────────────────────────────────────────
backbone_ids = {
    id(p) for p in raw_model.backbone.parameters()
} if hasattr(raw_model, "backbone") else set()

head_params = [
    p for p in raw_model.parameters()
    if id(p) not in backbone_ids
]

head_total = sum(p.numel() for p in head_params)
head_trainable = sum(
    p.numel() for p in head_params
    if p.requires_grad
)

print(
    f"Head / Neck             : "
    f"{head_total:,} "
    f"(trainables: {head_trainable:,}) "
    f"[{100 * head_total / total:.2f}%]"
)

print("-" * 65)


# ─────────────────────────────────────────────
# DÉTAIL DES SOUS-MODULES DIRECTS
# ─────────────────────────────────────────────
print()
print("🔍 DÉTAIL DES SOUS-MODULES")
print("-" * 65)

for name, module in raw_model.named_children():
    n_total, n_trainable = count_params(module)

    print(
        f"{name:<25} : "
        f"{n_total:>12,} "
        f"| trainable: {n_trainable:>12,} "
        f"| {100 * n_total / total:>6.2f}%"
    )

print("=" * 65)

## 8 — Lancer l'entraînement sur **2 GPU**


In [ ]:
import os
import sys
import subprocess
import torch
from pathlib import Path

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")
TRAIN_SCRIPT = PROJECT / "train_plankeye_ddp.py"
STORE_FILE = PROJECT / "ddp_filestore"

if not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(TRAIN_SCRIPT)

if torch.cuda.device_count() < 2:
    raise RuntimeError("2 GPU requis. Active GPU T4 x2.")

if STORE_FILE.exists():
    STORE_FILE.unlink()

print("Lancement DDP FileStore : GPU 0 + GPU 1")

base_env = os.environ.copy()
base_env["OMP_NUM_THREADS"] = "2"
base_env["PYTHONUNBUFFERED"] = "1"
base_env["NCCL_SOCKET_IFNAME"] = "lo"
base_env["NCCL_SOCKET_FAMILY"] = "AF_INET"
base_env["NCCL_DEBUG"] = "WARN"

processes = []
for rank in range(2):
    env = base_env.copy()
    env["RANK"] = str(rank)
    env["LOCAL_RANK"] = str(rank)
    env["WORLD_SIZE"] = "2"
    process = subprocess.Popen([sys.executable, str(TRAIN_SCRIPT)], cwd=str(PROJECT), env=env)
    processes.append(process)

return_codes = [p.wait() for p in processes]
if any(code != 0 for code in return_codes):
    raise RuntimeError(f"Un processus DDP a échoué : {return_codes}")

print("Entraînement terminé correctement.")

In [ ]:
!nvidia-smi


In [ ]:
!kill -9 <PID>

## 9 — Sauvegarde **manuelle persistante** des checkpoints V5


In [ ]:
from pathlib import Path
import shutil
import json
import subprocess
import socket
import torch

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")
UPLOAD_DIR = Path("/kaggle/working/chekpoints-backbone18_persistent")
KAGGLE_DATASET_ID = "max778/chekpoints-backbone18"

BEST = PROJECT / "best_checkpoint_v6.pt"
LAST = PROJECT / "last_checkpoint_v6.pt"

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST, UPLOAD_DIR / BEST.name)
shutil.copy2(LAST, UPLOAD_DIR / LAST.name)

ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
epoch = int(ckpt.get("epoch", -1))
metadata = {
    "title": "chekpoints-backbone18",
    "id": KAGGLE_DATASET_ID,
    "licenses": [{"name": "other"}],
    "isPrivate": True,
}
with open(UPLOAD_DIR / "dataset-metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

kaggle_exe = shutil.which("kaggle")
cmd = [kaggle_exe, "datasets", "version", "-p", str(UPLOAD_DIR), "-m", f"Checkpoint V5 epoch {epoch}", "--delete-old-versions"]
result = subprocess.run(cmd, text=True, capture_output=True, check=False)

if result.returncode == 0:
    print("\n✅ Nouvelle version persistante sauvegardée.")
else:
    create_result = subprocess.run([kaggle_exe, "datasets", "create", "-p", str(UPLOAD_DIR)], text=True, capture_output=True, check=False)
    if create_result.returncode != 0:
        raise RuntimeError("Sauvegarde persistante Kaggle impossible.")
    print("\n✅ Dataset de checkpoints créé.")

In [ ]:
from pathlib import Path
import shutil
import subprocess
import json
import torch

PROJECT = Path("/kaggle/working/PlankEyev2_multipieces")
PERSIST = Path("/kaggle/working/chekpoints-backbone18_persistent")
DATASET_ID = "max778/chekpoints-backbone18"

# Recrée le dossier avec les derniers checkpoints
if PERSIST.exists():
    shutil.rmtree(PERSIST)
PERSIST.mkdir(parents=True)

# Copie les checkpoints
for name in [
    "best_checkpoint_v6.pt",
    "last_checkpoint_v6.pt",
]:
    src = PROJECT / name
    if src.exists():
        shutil.copy2(src, PERSIST / name)

# Récupère l'epoch
ckpt = torch.load(
    PROJECT / "last_checkpoint_v6.pt",
    map_location="cpu",
    weights_only=False
)
epoch = int(ckpt.get("epoch", -1))

# Metadata
with open(PERSIST / "dataset-metadata.json", "w") as f:
    json.dump({
        "title": "chekpoints-backbone18",
        "id": DATASET_ID,
        "licenses": [{"name": "other"}],
        "isPrivate": True
    }, f, indent=2)

# Nouvelle VERSION Kaggle
# IMPORTANT : aucune suppression des anciennes versions
kaggle = shutil.which("kaggle")

result = subprocess.run([
    kaggle,
    "datasets", "version",
    "-p", str(PERSIST),
    "-m", f"PlankEye V5 - epoch {epoch}"
], capture_output=True, text=True)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("❌ Échec sauvegarde Kaggle")

print(f"✅ Historique persistant sauvegardé — epoch {epoch}")
print(f"📦 {DATASET_ID}")
print("📚 Anciennes versions conservées")

In [ ]:
from pathlib import Path

print("Datasets disponibles :\n")

for p in Path("/kaggle/input").iterdir():
    print("📁", p)

print("\nRecherche des checkpoints :\n")

for p in Path("/kaggle/input").rglob("*.pt"):
    print("✅", p)

In [ ]:
from pathlib import Path
import torch

ckpt_path = Path(
    "/kaggle/input/datasets/max778/chekpoints-backbone18/best_checkpoint_v6.pt"
)

print("Checkpoint :", ckpt_path)
print("Existe :", ckpt_path.exists())
print("Taille :", ckpt_path.stat().st_size / 1024**2, "MB")

ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("\nType :", type(ckpt))

if isinstance(ckpt, dict):
    print("\nClés du checkpoint :")
    for k in ckpt.keys():
        print(" -", k)

    if "histories" in ckpt:
        print("\n✅ HISTORIES TROUVÉES !")

        histories = ckpt["histories"]

        print("\nContenu :")
        for k, v in histories.items():
            print(f" - {k}: {len(v)} valeurs")

    else:
        print("\n❌ Pas de 'histories' dans best_checkpoint_v6.pt")

In [ ]:
ckpt = torch.load(
    "/kaggle/input/datasets/max778/chekpoints-backbone18/best_checkpoint_v6.pt",
    map_location="cpu",
    weights_only=False
)

histories = ckpt["histories"]

print(histories)